In [ ]:
from pathlib import Path
from contextlib import nullcontext
from dataclasses import dataclass
from functools import lru_cache
import argparse
import csv
import json
import math
import os
import pickle
import random
import sys
import time
from typing import Dict, Iterable, List, Optional, Tuple

import numpy as np
import torch
import torch.distributed as dist
import torch.nn.functional as F
from torch import nn
from torch.nn.functional import scaled_dot_product_attention
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import DataLoader, Dataset
from torch.utils.data.distributed import DistributedSampler
from tqdm.std import tqdm
from transformers import AutoTokenizer


def make_tqdm(*args, **kwargs):
    kwargs.setdefault("file", sys.stdout)
    kwargs.setdefault("dynamic_ncols", True)
    kwargs.setdefault("mininterval", 5)
    kwargs.setdefault("leave", True)
    return tqdm(*args, **kwargs)


def parse_runtime_overrides() -> Dict[str, Optional[str]]:
    if "ipykernel" in sys.modules:
        return {}

    parser = argparse.ArgumentParser(description="Train the TRM reranker with plain PyTorch.")
    parser.add_argument("--profile", choices=["smoke", "full", "full_all_dev"], default=None)
    parser.add_argument("--output-root", default=None)
    parser.add_argument("--resume-from-checkpoint", default=None)
    parser.add_argument("--run-id", default=None)
    args, _ = parser.parse_known_args()

    return {key: value for key, value in vars(args).items() if value is not None}


RUNTIME_OVERRIDES = parse_runtime_overrides()


PREP_MANIFEST_PATH = Path("<path/to/prep_manifest.json>")
# Path to prep_manifest.json with tokenizer and dataset artifacts.


RUN_DATA_MANIFEST_PATH = Path("<path/to/run_data_manifest.json>")
# Path to run_data_manifest.json produced by run-data cache preparation.


OUTPUT_ROOT = Path("<path/to/output_dir>")
# Directory where logs, checkpoints, and evaluation files are written.


ABLATION_OUTPUT_DIRNAME = "1_epoch"
IS_ABLATION_NOTEBOOK = False

FREEZE_TOKEN_EMBEDDINGS = False
FREEZE_SEGMENT_EMBEDDINGS = False


RUN_PROFILE = RUNTIME_OVERRIDES.get("profile") or "full"

PROFILE_SETTINGS = {
    "smoke": {
        "train_triples_sample": 100_000,
        "epochs": 1,
        "max_train_steps": 20,
        "devices": 1,
        "grad_accum_steps": 1,
        "use_ddp": False,
    },
    "full": {
        "train_triples_sample": 398_000_000,
        "epochs": 1,
        "max_train_steps": None,
        "devices": 2,
        "grad_accum_steps": 1,
        "use_ddp": True,
    },
    "full_all_dev": {
        "train_triples_sample": 1_000_000,
        "epochs": 5,
        "max_train_steps": None,
        "devices": 2,
        "grad_accum_steps": 1,
        "use_ddp": True,
    },
}

if RUN_PROFILE not in PROFILE_SETTINGS:
    raise ValueError(
        f"Unsupported RUN_PROFILE={RUN_PROFILE!r}. "
        f"Expected one of {sorted(PROFILE_SETTINGS)}"
    )

PROFILE = dict(PROFILE_SETTINGS[RUN_PROFILE])


SEED = 13
TARGET_TOTAL_EPOCHS = 1
RUN_EPOCH_FRACTION = 0.035
RUN_TRAIN_STEPS = None

CHECKPOINT_EPOCH_FRACTION = 0.005
CHECKPOINT_EVERY_N_STEPS = None
DEV_EVAL_QUERY_LIMIT = 500
TQDM_POSTFIX_EVERY_N_STEPS = 200

RESUME_FROM_CHECKPOINT = True
RESUME_CHECKPOINT_PATH = Path("<path/to/resume_checkpoint.pt>")
# Required only when RESUME_FROM_CHECKPOINT=True.


RUN_EPOCH_DEV_EVAL_ON_PARTIAL = False


ALLOW_RESUME_LR_OVERRIDE = False


TRAIN_TRIPLES_SAMPLE = int(PROFILE["train_triples_sample"])
EPOCHS = int(TARGET_TOTAL_EPOCHS)
PROFILE["epochs"] = int(TARGET_TOTAL_EPOCHS)

MAX_TRAIN_STEPS = None


if RUNTIME_OVERRIDES.get("resume_from_checkpoint"):
    RESUME_FROM_CHECKPOINT = True
    RESUME_CHECKPOINT_PATH = Path(RUNTIME_OVERRIDES["resume_from_checkpoint"])


DEVICES = int(PROFILE["devices"])
GRAD_ACCUM_STEPS = int(PROFILE["grad_accum_steps"])
USE_DDP = bool(PROFILE["use_ddp"]) and DEVICES > 1

if GRAD_ACCUM_STEPS <= 0:
    raise ValueError("GRAD_ACCUM_STEPS must be >= 1")


PER_DEVICE_BATCH_SIZE = 512
EVAL_BATCH_SIZE = 512


LEARNING_RATE = 2e-4

WEIGHT_DECAY = 0.01
MAX_GRAD_NORM = 1.0
WARMUP_RATIO = 0.06


PRECISION = "bf16-mixed"

NUM_WORKERS = 0

EXPERIMENT_NAME_PREFIX = "trm_reranker_ablation_frozen_token_embeddings"


HIDDEN_SIZE = 512
NUM_HEADS = 8
L_LAYERS = 2
H_CYCLES = 2
L_CYCLES = 4
HALT_MAX_STEPS = 1
HALT_EXPLORATION_PROB = 0.0
POS_ENCODINGS = "rope"
EXPANSION = 4.0

TOKENIZER_NAME = "bert-base-uncased"
SEQ_LEN = None
MAX_QUERY_LEN = None
MAX_DOC_LEN = None


REPO_ROOT = Path.cwd().resolve()

def is_empty_path(path: Path) -> bool:
    return str(path).strip() in {"", "."}


def display_path(path: Path) -> str:
    return "(not configured)" if is_empty_path(path) else str(path)


OUTPUT_ROOT = Path(OUTPUT_ROOT).expanduser() if not is_empty_path(OUTPUT_ROOT) else Path("")


RUN_ID = (
    RUNTIME_OVERRIDES.get("run_id")
    or os.environ.get("TRM_RUN_ID")
    or os.environ.get("TORCHELASTIC_RUN_ID")
    or time.strftime("%Y%m%d-%H%M%S")
)

EXPERIMENT_NAME = f"{EXPERIMENT_NAME_PREFIX}_{RUN_PROFILE}_{RUN_ID}"


print("=" * 80)
print("Training config")
print("=" * 80)
print(f"RUN_PROFILE:                  {RUN_PROFILE}")
print(f"TARGET_TOTAL_EPOCHS:          {TARGET_TOTAL_EPOCHS}")
print(f"EPOCHS:                       {EPOCHS}")
print(f"RUN_EPOCH_FRACTION:           {RUN_EPOCH_FRACTION}")
print(f"RUN_TRAIN_STEPS:              {RUN_TRAIN_STEPS}")
print(f"CHECKPOINT_EPOCH_FRACTION:    {CHECKPOINT_EPOCH_FRACTION}")
print(f"CHECKPOINT_EVERY_N_STEPS:     {CHECKPOINT_EVERY_N_STEPS}")
print(f"DEV_EVAL_QUERY_LIMIT:         {DEV_EVAL_QUERY_LIMIT}")
print(f"RESUME_FROM_CHECKPOINT:       {RESUME_FROM_CHECKPOINT}")
print(f"RESUME_CHECKPOINT_PATH:       {display_path(RESUME_CHECKPOINT_PATH)}")
print(f"RUN_EPOCH_DEV_EVAL_ON_PARTIAL:{RUN_EPOCH_DEV_EVAL_ON_PARTIAL}")
print(f"TRAIN_TRIPLES_SAMPLE:         {TRAIN_TRIPLES_SAMPLE}")
print(f"PER_DEVICE_BATCH_SIZE:        {PER_DEVICE_BATCH_SIZE}")
print(f"GRAD_ACCUM_STEPS:             {GRAD_ACCUM_STEPS}")
print(f"DEVICES:                      {DEVICES}")
print(f"USE_DDP:                      {USE_DDP}")
print(f"LEARNING_RATE:                {LEARNING_RATE}")
print(f"OUTPUT_ROOT:                  {display_path(OUTPUT_ROOT)}")
print(f"EXPERIMENT_NAME:              {EXPERIMENT_NAME}")
print("=" * 80)


In [ ]:
def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def load_pickle(path: Path):
    with path.open('rb') as handle:
        return pickle.load(handle)


def require_configured_path(path: Path, name: str) -> Path:
    del name
    return Path(path).expanduser()


def require_file(path: Path, name: str) -> Path:
    return require_configured_path(path, name).resolve()


def require_dir(path: Path, name: str) -> Path:
    return require_configured_path(path, name).resolve()


def require_output_dir(path: Path, name: str) -> Path:
    resolved = require_configured_path(path, name).resolve()
    resolved.mkdir(parents=True, exist_ok=True)
    return resolved


def resolve_relative_or_absolute_path(value: str, base_dir: Path) -> Path:
    path = Path(str(value)).expanduser()
    if path.is_absolute():
        return path.resolve()
    return (base_dir / path).resolve()


def get_manifest_mapping(manifest: dict, section_name: str) -> Dict[str, object]:
    value = manifest.get(section_name) or {}
    if not isinstance(value, dict):
        return {}
    return value


def manifest_value_candidates(primary_key: str, aliases: Optional[List[str]] = None) -> List[str]:
    keys = [primary_key]
    if aliases:
        keys.extend(aliases)
    unique: List[str] = []
    seen = set()
    for key in keys:
        if not key or key in seen:
            continue
        seen.add(key)
        unique.append(key)
    return unique


def get_manifest_mapping_value(mapping: Dict[str, object], keys: List[str]):
    for key in keys:
        value = mapping.get(key)
        if value:
            return value, key
    return None, None


def validate_prep_manifest(manifest: dict) -> None:
    schema_version = int(manifest.get('schema_version', 0))
    if schema_version < 4:
        raise ValueError(
            'Prep manifest schema_version is too old for the dataset-level cache workflow. '
            'Re-run 00_prepare_data_cache.ipynb to build the sharded full-collection passage cache.'
        )

    required_artifact_keys = {
        'train_query_tokens_pkl': [],
        'dev_query_tokens_pkl': [],
        'passage_token_shards_dir': ['passage_tokens_shards_dir'],
        'passage_token_shards_index_json': ['passage_tokens_store_index_json'],
        'passage_token_store_stats_json': ['passage_tokens_store_stats_json'],
        'dev_candidates_pkl': [],
        'dev_qrels_pkl': [],
    }
    artifacts = get_manifest_mapping(manifest, 'artifacts')
    missing_artifacts = [
        key
        for key, aliases in required_artifact_keys.items()
        if get_manifest_mapping_value(artifacts, manifest_value_candidates(key, aliases))[0] is None
    ]
    if missing_artifacts:
        raise ValueError(
            'Prep manifest is missing required dataset-level artifacts. '
            f'Re-run 00_prepare_data_cache.ipynb. Missing keys: {missing_artifacts}. '
            f'Available artifact keys: {sorted(artifacts)}'
        )


def is_dist_available_and_initialized() -> bool:
    return dist.is_available() and dist.is_initialized()


def get_rank() -> int:
    if is_dist_available_and_initialized():
        return dist.get_rank()
    return int(os.environ.get("RANK", "0"))


def get_local_rank() -> int:
    return int(os.environ.get("LOCAL_RANK", "0"))


def get_world_size() -> int:
    if is_dist_available_and_initialized():
        return dist.get_world_size()
    if USE_DDP and int(os.environ.get("WORLD_SIZE", "1")) > 1:
        return int(os.environ.get("WORLD_SIZE", "1"))
    return 1


def is_main_process() -> bool:
    return get_rank() == 0


def ddp_barrier() -> None:
    if is_dist_available_and_initialized():
        dist.barrier()


def reduce_mean(tensor: torch.Tensor) -> torch.Tensor:
    if is_dist_available_and_initialized():
        reduced = tensor.detach().clone()
        dist.all_reduce(reduced, op=dist.ReduceOp.SUM)
        reduced /= dist.get_world_size()
        return reduced
    return tensor


def unwrap_model(model):
    return model.module if hasattr(model, "module") else model


def resolve_precision(requested_precision: str, device_kind: str) -> str:
    if requested_precision == "bf16-mixed":
        if device_kind == "cuda" and torch.cuda.is_available() and torch.cuda.is_bf16_supported():
            return requested_precision
        if is_main_process():
            print(f'Falling back from precision={requested_precision!r} to "32-true" because bf16 is not available in this runtime.')
        return "32-true"
    if requested_precision == "16-mixed" and device_kind != "cuda":
        if is_main_process():
            print(f'Falling back from precision={requested_precision!r} to "32-true" because fp16 autocast needs CUDA.')
        return "32-true"
    return requested_precision


def setup_distributed() -> torch.device:
    world_size = get_world_size()
    if USE_DDP and world_size > 1:
        if not torch.cuda.is_available():
            raise RuntimeError("DDP was requested, but CUDA is not available.")
        if not is_dist_available_and_initialized():
            dist.init_process_group(backend="nccl")
        local_rank = get_local_rank()
        torch.cuda.set_device(local_rank)
        return torch.device("cuda", local_rank)
    return torch.device("cuda", 0) if torch.cuda.is_available() else torch.device("cpu")


def cleanup_distributed() -> None:
    if is_dist_available_and_initialized():
        dist.destroy_process_group()


def resolve_run_manifest_artifact_path(
    run_manifest: dict,
    key: str,
    base_dir: Path,
    aliases: Optional[List[str]] = None,
) -> Path:
    artifacts = get_manifest_mapping(run_manifest, "artifacts")
    value, _ = get_manifest_mapping_value(artifacts, manifest_value_candidates(key, aliases))
    if value is None:
        raise KeyError(
            f"Run-data manifest is missing required artifact key {key!r}. "
            f"Available artifact keys: {sorted(artifacts)}"
        )
    return resolve_relative_or_absolute_path(str(value), base_dir)


def validate_run_data_manifest(run_manifest: dict) -> None:
    schema_version = int(run_manifest.get("schema_version", 0))
    if schema_version < 1:
        raise ValueError(
            "run_data_manifest.json has an unsupported schema_version. "
            "Re-run 01_prepare_run_data_cache.ipynb."
        )
    required_top_level = [
        "base_prep_manifest_path",
        "base_artifact_dir",
        "train_triples_sample",
        "seed",
        "dev_eval_mode",
        "dev_eval_seed",
        "epoch_dev_mode_label",
        "run_final_full_dev",
        "counts",
        "artifacts",
    ]
    missing_top_level = [key for key in required_top_level if run_manifest.get(key) is None]
    if missing_top_level:
        raise ValueError(
            "run_data_manifest.json is missing required top-level keys. "
            f"Missing: {missing_top_level}"
        )

    dev_eval_mode = str(run_manifest.get("dev_eval_mode"))
    if dev_eval_mode == "quick_count" and run_manifest.get("dev_eval_query_count") is None:
        raise ValueError("run_data_manifest.json is missing dev_eval_query_count for quick_count mode.")
    if dev_eval_mode == "quick_fraction" and run_manifest.get("dev_eval_fraction") is None:
        raise ValueError("run_data_manifest.json is missing dev_eval_fraction for quick_fraction mode.")
    if dev_eval_mode not in {"quick_count", "quick_fraction", "full"}:
        raise ValueError(f"run_data_manifest.json has unsupported dev_eval_mode={dev_eval_mode!r}")

    required_artifact_keys = [
        "sampled_train_triples_tsv",
        "train_query_tokens_pkl",
        "train_passage_tokens_pkl",
        "dev_query_tokens_pkl",
        "epoch_dev_candidates_pkl",
        "epoch_dev_qrels_pkl",
        "passage_token_shards_dir",
        "passage_token_shards_index_json",
        "passage_token_store_stats_json",
    ]
    if bool(run_manifest.get("run_final_full_dev", True)):
        required_artifact_keys.extend(["final_dev_candidates_pkl", "final_dev_qrels_pkl"])
    artifacts = get_manifest_mapping(run_manifest, "artifacts")
    missing_artifacts = [key for key in required_artifact_keys if artifacts.get(key) is None]
    if missing_artifacts:
        raise ValueError(
            "run_data_manifest.json is missing required artifact entries. "
            f"Missing: {missing_artifacts}. Available: {sorted(artifacts)}"
        )


def validate_run_data_compatibility(run_manifest: dict, prep_manifest: dict) -> None:
    mismatches = {}
    prep_expected = {
        "tokenizer_name": str(prep_manifest.get("tokenizer_name", "")),
        "seq_len": int(prep_manifest.get("seq_len", -1)),
        "max_query_len": int(prep_manifest.get("max_query_len", -1)),
        "max_doc_len": int(prep_manifest.get("max_doc_len", -1)),
    }
    run_actual = {
        "tokenizer_name": str(run_manifest.get("tokenizer_name", "")),
        "seq_len": int(run_manifest.get("seq_len", -1)),
        "max_query_len": int(run_manifest.get("max_query_len", -1)),
        "max_doc_len": int(run_manifest.get("max_doc_len", -1)),
    }
    for key, expected_value in prep_expected.items():
        if run_actual[key] != expected_value:
            mismatches[key] = {"expected": expected_value, "actual": run_actual[key]}

    run_config_expected = {
        "train_triples_sample": int(TRAIN_TRIPLES_SAMPLE),
        "seed": int(SEED),
    }
    run_config_actual = {
        "train_triples_sample": int(run_manifest.get("train_triples_sample", -1)),
        "seed": int(run_manifest.get("seed", -1)),
    }
    for key, expected_value in run_config_expected.items():
        if run_config_actual[key] != expected_value:
            mismatches[key] = {"expected": expected_value, "actual": run_config_actual[key]}

    if mismatches:
        raise ValueError(
            "run_data_manifest.json is incompatible with the prep manifest or current training run settings. "
            f"Mismatches: {mismatches}. Re-run 01_prepare_run_data_cache.ipynb with matching train sample settings."
        )


def load_passage_token_shard_index(index_path: Path) -> dict:
    return json.loads(index_path.read_text())


def build_passage_token_subset_loader(
    index_path: Path,
    artifact_dir: Path,
    shards_dir_path: Optional[Path] = None,
    shard_cache_size: int = 512,
):
    index = load_passage_token_shard_index(index_path)
    if index.get("format") != "sharded_flat_token_arrays_v1":
        raise ValueError(
            f"Unsupported passage token store format: {index.get('format')!r}. "
            "This notebook expects a sharded flat token store described by passage_token_shards_index_json."
        )

    shard_entries = index.get("shards") or []
    if not shard_entries:
        raise ValueError(f"Passage token shard index is missing shard entries: {index_path}")

    shard_size = int(index["shard_size"])
    shards_by_id = {int(entry["shard_id"]): entry for entry in shard_entries}
    resolved_shards_dir = Path(shards_dir_path).resolve() if shards_dir_path is not None else None

    def resolve_shard_path(shard_entry: dict) -> Path:
        raw_path_value = shard_entry.get("path")
        if raw_path_value:
            return require_file(
                resolve_relative_or_absolute_path(str(raw_path_value), artifact_dir),
                f"passage shard {shard_entry.get('shard_id')}",
            )

        filename = shard_entry.get("filename") or shard_entry.get("file_name") or shard_entry.get("basename") or shard_entry.get("name")
        if filename and resolved_shards_dir is not None:
            return require_file(
                resolved_shards_dir / str(filename),
                f"passage shard {shard_entry.get('shard_id')}",
            )

        raise FileNotFoundError(
            "Passage token shard entry does not provide a path or filename: "
            f"shard_id={shard_entry.get('shard_id')}"
        )

    @lru_cache(maxsize=shard_cache_size)
    def load_shard(shard_id: int):
        shard_entry = shards_by_id.get(int(shard_id))
        if shard_entry is None:
            raise KeyError(f"No passage token shard found for shard_id={shard_id}")
        shard_path = resolve_shard_path(shard_entry)
        with np.load(shard_path, allow_pickle=False) as shard_data:
            shard_pids = shard_data["pid"]
            shard_offsets = shard_data["offsets"]
            shard_token_ids = shard_data["token_ids"]
        pid_lookup = {int(pid): idx for idx, pid in enumerate(shard_pids.tolist())}
        return {
            "pid": shard_pids,
            "offsets": shard_offsets,
            "token_ids": shard_token_ids,
            "pid_lookup": pid_lookup,
        }

    def get_passage_tokens(pid: int) -> List[int]:
        shard = load_shard(int(pid) // shard_size)
        pid_idx = shard["pid_lookup"].get(int(pid))
        if pid_idx is None:
            raise KeyError(f"Missing cached passage tokens for pid={pid}")
        start = int(shard["offsets"][pid_idx])
        end = int(shard["offsets"][pid_idx + 1])
        return shard["token_ids"][start:end].tolist()

    def load_passage_tokens_for_subset(pid_list: Iterable[int]) -> Dict[int, List[int]]:
        unique_pids = sorted({int(pid) for pid in pid_list})
        pids_by_shard: Dict[int, List[int]] = {}
        for pid in unique_pids:
            pids_by_shard.setdefault(int(pid) // shard_size, []).append(int(pid))

        subset: Dict[int, List[int]] = {}
        with make_tqdm(
            total=len(unique_pids),
            desc="Load cached train passages",
            disable=not is_main_process(),
        ) as pbar:
            for shard_id in sorted(pids_by_shard):
                shard = load_shard(shard_id)
                for pid in pids_by_shard[shard_id]:
                    pid_idx = shard["pid_lookup"].get(pid)
                    if pid_idx is None:
                        raise KeyError(f"Missing cached passage tokens for pid={pid}")
                    start = int(shard["offsets"][pid_idx])
                    end = int(shard["offsets"][pid_idx + 1])
                    subset[pid] = shard["token_ids"][start:end].tolist()
                    pbar.update(1)
        return subset

    return index, get_passage_tokens, load_passage_tokens_for_subset


seed_everything(SEED)

AVAILABLE_CUDA_DEVICES = torch.cuda.device_count() if torch.cuda.is_available() else 0
WORLD_SIZE_ENV = int(os.environ.get("WORLD_SIZE", "1"))
DEVICE_KIND = "cuda" if torch.cuda.is_available() else "cpu"
SHOULD_USE_DDP = USE_DDP and WORLD_SIZE_ENV > 1
if USE_DDP and WORLD_SIZE_ENV == 1 and is_main_process():
    print("USE_DDP=True but WORLD_SIZE=1. Run with torchrun for real multi-GPU training.")
    print("Falling back to single-process training.")
if DEVICES > AVAILABLE_CUDA_DEVICES and AVAILABLE_CUDA_DEVICES > 0 and is_main_process():
    print(f"Requested DEVICES={DEVICES}, but only {AVAILABLE_CUDA_DEVICES} CUDA devices are visible in this runtime.")

EFFECTIVE_PRECISION = resolve_precision(PRECISION, DEVICE_KIND)
MODEL_FORWARD_DTYPE = "bfloat16" if EFFECTIVE_PRECISION == "bf16-mixed" else "float32"
ACCELERATOR = "gpu" if DEVICE_KIND == "cuda" else "cpu"
TRAINER_DEVICES = WORLD_SIZE_ENV if SHOULD_USE_DDP else (min(max(1, DEVICES), AVAILABLE_CUDA_DEVICES) if ACCELERATOR == "gpu" else 1)
STRATEGY = "ddp" if SHOULD_USE_DDP else "single_process"
OUTPUT_ROOT = require_output_dir(OUTPUT_ROOT, "OUTPUT_ROOT")

RUN_DIR = OUTPUT_ROOT / "trm_reranker_mvp" / "runs" / EXPERIMENT_NAME
CHECKPOINT_DIR = RUN_DIR / "checkpoints"
LOG_DIR = RUN_DIR / "logs"
EVAL_DIR = RUN_DIR / "eval"
EPOCH_EVAL_DIR = EVAL_DIR / "epochs"
STEP_EVAL_DIR = EVAL_DIR / "steps"
FINAL_EVAL_DIR = EVAL_DIR / "final"
TRAIN_LOG_PATH = LOG_DIR / "train_metrics.csv"
TRAINING_CONFIG_PATH = RUN_DIR / "training_config.json"
FIT_SUMMARY_PATH = RUN_DIR / "fit_summary.json"
RUN_ARTIFACTS_PATH = RUN_DIR / "run_artifacts.json"
EPOCH_SUMMARIES_PATH = LOG_DIR / "epoch_summaries.json"
DEV_METRICS_BY_EPOCH_PATH = LOG_DIR / "dev_metrics_by_epoch.csv"
DEV_METRICS_BY_STEP_PATH = LOG_DIR / "dev_metrics_by_step.csv"
LAST_CHECKPOINT_PATH = CHECKPOINT_DIR / "last_checkpoint.pt"
BEST_TRAIN_LOSS_CHECKPOINT_PATH = CHECKPOINT_DIR / "best_train_loss.pt"
BEST_MRR_CHECKPOINT_PATH = CHECKPOINT_DIR / "best_mrr.pt"
BEST_DEV_MRR10_CHECKPOINT_PATH = CHECKPOINT_DIR / "best_dev_mrr10.pt"
FINAL_RUN_PATH = FINAL_EVAL_DIR / "final_dev_best.run"
FINAL_METRICS_PATH = FINAL_EVAL_DIR / "final_dev_metrics.json"
RUN_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)
EVAL_DIR.mkdir(parents=True, exist_ok=True)
EPOCH_EVAL_DIR.mkdir(parents=True, exist_ok=True)
STEP_EVAL_DIR.mkdir(parents=True, exist_ok=True)
FINAL_EVAL_DIR.mkdir(parents=True, exist_ok=True)

RUN_DATA_MANIFEST_PATH = require_file(RUN_DATA_MANIFEST_PATH, "RUN_DATA_MANIFEST_PATH")
if RESUME_FROM_CHECKPOINT:
    RESUME_CHECKPOINT_PATH = require_file(RESUME_CHECKPOINT_PATH, "RESUME_CHECKPOINT_PATH")

run_data_manifest = json.loads(RUN_DATA_MANIFEST_PATH.read_text())
validate_run_data_manifest(run_data_manifest)
RUN_DATA_CACHE_DIR = RUN_DATA_MANIFEST_PATH.parent.resolve()
if is_empty_path(PREP_MANIFEST_PATH):
    PREP_MANIFEST_PATH = resolve_relative_or_absolute_path(run_data_manifest["base_prep_manifest_path"], RUN_DATA_CACHE_DIR)
PREP_MANIFEST_PATH = require_file(PREP_MANIFEST_PATH, "PREP_MANIFEST_PATH")
ARTIFACT_DIR = require_dir(PREP_MANIFEST_PATH.parent, "PREP_MANIFEST_PATH.parent")
manifest = json.loads(PREP_MANIFEST_PATH.read_text())
validate_prep_manifest(manifest)
validate_run_data_compatibility(run_data_manifest, manifest)

RUN_FINAL_FULL_DEV = bool(run_data_manifest.get("run_final_full_dev", True))
RUN_DATA_DEV_EVAL_MODE = str(run_data_manifest.get("dev_eval_mode"))
RUN_DATA_DEV_EVAL_FRACTION = run_data_manifest.get("dev_eval_fraction")
RUN_DATA_DEV_EVAL_QUERY_COUNT = run_data_manifest.get("dev_eval_query_count")
RUN_DATA_DEV_EVAL_SEED = int(run_data_manifest.get("dev_eval_seed", SEED))
RUN_DATA_EPOCH_DEV_MODE_LABEL = str(run_data_manifest.get("epoch_dev_mode_label") or RUN_DATA_DEV_EVAL_MODE)

TOKENIZER_NAME = str(manifest.get("tokenizer_name", run_data_manifest.get("tokenizer_name", TOKENIZER_NAME)))
SEQ_LEN = int(manifest["seq_len"])
MAX_QUERY_LEN = int(manifest["max_query_len"])
MAX_DOC_LEN = int(manifest["max_doc_len"])

tokenizer_local_value = manifest.get("tokenizer_local_path", "tokenizer")
tokenizer_local_path = resolve_relative_or_absolute_path(tokenizer_local_value, ARTIFACT_DIR)
sampled_train_triples_path = require_file(
    resolve_run_manifest_artifact_path(run_data_manifest, "sampled_train_triples_tsv", RUN_DATA_CACHE_DIR),
    "sampled_train_triples_tsv",
)
train_query_tokens_path = require_file(
    resolve_run_manifest_artifact_path(run_data_manifest, "train_query_tokens_pkl", RUN_DATA_CACHE_DIR),
    "train_query_tokens_pkl",
)
train_passage_tokens_path = require_file(
    resolve_run_manifest_artifact_path(run_data_manifest, "train_passage_tokens_pkl", RUN_DATA_CACHE_DIR),
    "train_passage_tokens_pkl",
)
dev_query_tokens_path = require_file(
    resolve_run_manifest_artifact_path(run_data_manifest, "dev_query_tokens_pkl", ARTIFACT_DIR),
    "dev_query_tokens_pkl",
)
epoch_dev_candidates_path = require_file(
    resolve_run_manifest_artifact_path(run_data_manifest, "epoch_dev_candidates_pkl", RUN_DATA_CACHE_DIR),
    "epoch_dev_candidates_pkl",
)
epoch_dev_qrels_path = require_file(
    resolve_run_manifest_artifact_path(run_data_manifest, "epoch_dev_qrels_pkl", RUN_DATA_CACHE_DIR),
    "epoch_dev_qrels_pkl",
)
final_dev_candidates_path = (
    require_file(resolve_run_manifest_artifact_path(run_data_manifest, "final_dev_candidates_pkl", ARTIFACT_DIR), "final_dev_candidates_pkl")
    if RUN_FINAL_FULL_DEV else Path("")
)
final_dev_qrels_path = (
    require_file(resolve_run_manifest_artifact_path(run_data_manifest, "final_dev_qrels_pkl", ARTIFACT_DIR), "final_dev_qrels_pkl")
    if RUN_FINAL_FULL_DEV else Path("")
)
passage_token_shards_dir_path = require_dir(
    resolve_run_manifest_artifact_path(run_data_manifest, "passage_token_shards_dir", ARTIFACT_DIR, aliases=["passage_tokens_shards_dir"]),
    "passage_token_shards_dir",
)
passage_token_shards_index_path = require_file(
    resolve_run_manifest_artifact_path(run_data_manifest, "passage_token_shards_index_json", ARTIFACT_DIR, aliases=["passage_tokens_store_index_json"]),
    "passage_token_shards_index_json",
)
passage_token_store_stats_path = require_file(
    resolve_run_manifest_artifact_path(run_data_manifest, "passage_token_store_stats_json", ARTIFACT_DIR, aliases=["passage_tokens_store_stats_json"]),
    "passage_token_store_stats_json",
)

tokenizer_name = TOKENIZER_NAME
if tokenizer_local_path.exists():
    tokenizer = AutoTokenizer.from_pretrained(tokenizer_local_path, use_fast=True, local_files_only=True)
else:
    tokenizer = AutoTokenizer.from_pretrained(tokenizer_name, use_fast=True)

train_query_tokens = load_pickle(train_query_tokens_path)
train_passage_tokens = load_pickle(train_passage_tokens_path)
dev_query_tokens = load_pickle(dev_query_tokens_path)
epoch_dev_candidates_artifact = load_pickle(epoch_dev_candidates_path)
epoch_dev_qrels_artifact = load_pickle(epoch_dev_qrels_path)
final_dev_candidates = load_pickle(final_dev_candidates_path) if RUN_FINAL_FULL_DEV else None
final_dev_qrels = load_pickle(final_dev_qrels_path) if RUN_FINAL_FULL_DEV else None

passage_token_store_index, get_cached_passage_tokens, _load_passage_tokens_for_subset = build_passage_token_subset_loader(
    passage_token_shards_index_path,
    ARTIFACT_DIR,
    shards_dir_path=passage_token_shards_dir_path,
)

RUN_DATA_COUNTS = get_manifest_mapping(run_data_manifest, "counts")
sampled_train_triples_count = int(RUN_DATA_COUNTS.get("sampled_train_triples", 0))
sampled_train_queries_count = int(RUN_DATA_COUNTS.get("sampled_train_queries", len(train_query_tokens)))
sampled_train_passages_count = int(RUN_DATA_COUNTS.get("sampled_train_passages", len(train_passage_tokens)))
epoch_dev_queries_count = int(RUN_DATA_COUNTS.get("epoch_dev_queries", len(epoch_dev_candidates_artifact["qid_order"])))
epoch_dev_candidate_rows_count = int(RUN_DATA_COUNTS.get("epoch_dev_candidate_rows", len(epoch_dev_candidates_artifact["pid"])))
CLS_ID = tokenizer.cls_token_id
SEP_ID = tokenizer.sep_token_id
PAD_ID = tokenizer.pad_token_id
if CLS_ID is None or SEP_ID is None or PAD_ID is None:
    raise ValueError("Tokenizer must provide cls_token_id, sep_token_id, and pad_token_id")

model_config = {
    "batch_size": PER_DEVICE_BATCH_SIZE,
    "seq_len": SEQ_LEN,
    "vocab_size": len(tokenizer),
    "H_cycles": H_CYCLES,
    "L_cycles": L_CYCLES,
    "H_layers": L_LAYERS,
    "L_layers": L_LAYERS,
    "hidden_size": HIDDEN_SIZE,
    "expansion": EXPANSION,
    "num_heads": NUM_HEADS,
    "pos_encodings": POS_ENCODINGS,
    "halt_max_steps": HALT_MAX_STEPS,
    "halt_exploration_prob": HALT_EXPLORATION_PROB,
    "forward_dtype": MODEL_FORWARD_DTYPE,
    "no_ACT_continue": True,
    "num_segment_types": 3,
}

prep_summary = {
    "run_profile": RUN_PROFILE,
    "manifest_cache_tag": manifest.get("cache_tag"),
    "prep_manifest_path": str(PREP_MANIFEST_PATH),
    "run_data_manifest_path": str(RUN_DATA_MANIFEST_PATH),
    "artifact_dir": str(ARTIFACT_DIR),
    "run_data_cache_dir": str(RUN_DATA_CACHE_DIR),
    "output_root": str(OUTPUT_ROOT),
    "accelerator": ACCELERATOR,
    "requested_devices": DEVICES,
    "trainer_devices": TRAINER_DEVICES,
    "available_cuda_devices": AVAILABLE_CUDA_DEVICES,
    "strategy": STRATEGY,
    "precision": EFFECTIVE_PRECISION,
    "run_dir": str(RUN_DIR),
    "dev_eval_mode": RUN_DATA_DEV_EVAL_MODE,
    "dev_eval_fraction": RUN_DATA_DEV_EVAL_FRACTION,
    "dev_eval_query_count": RUN_DATA_DEV_EVAL_QUERY_COUNT,
    "dev_eval_seed": RUN_DATA_DEV_EVAL_SEED,
    "run_final_full_dev": RUN_FINAL_FULL_DEV,
    "sampled_train_triples_path": str(sampled_train_triples_path),
    "sampled_train_triples": sampled_train_triples_count,
    "sampled_train_queries": sampled_train_queries_count,
    "sampled_train_passages": sampled_train_passages_count,
    "epoch_dev_queries": epoch_dev_queries_count,
    "epoch_dev_candidate_rows": epoch_dev_candidate_rows_count,
    "epoch_dev_mode_label": RUN_DATA_EPOCH_DEV_MODE_LABEL,
    "passage_token_store_format": passage_token_store_index.get("format"),
    "passage_token_shard_count": int(passage_token_store_index.get("shard_count", len(passage_token_store_index.get("shards", [])))),
    "ablation_notebook": IS_ABLATION_NOTEBOOK,
    "ablation_variant": ABLATION_OUTPUT_DIRNAME,
    "freeze_token_embeddings": FREEZE_TOKEN_EMBEDDINGS,
    "freeze_segment_embeddings": FREEZE_SEGMENT_EMBEDDINGS,
}
run_artifacts = {
    "prep_manifest_path": str(PREP_MANIFEST_PATH),
    "run_data_manifest_path": str(RUN_DATA_MANIFEST_PATH),
    "artifact_dir": str(ARTIFACT_DIR),
    "run_data_cache_dir": str(RUN_DATA_CACHE_DIR),
    "output_root": str(OUTPUT_ROOT),
    "run_dir": str(RUN_DIR),
    "checkpoint_dir": str(CHECKPOINT_DIR),
    "log_dir": str(LOG_DIR),
    "eval_dir": str(EVAL_DIR),
    "epoch_eval_dir": str(EPOCH_EVAL_DIR),
    "step_eval_dir": str(STEP_EVAL_DIR),
    "final_eval_dir": str(FINAL_EVAL_DIR),
    "dev_eval_mode": RUN_DATA_DEV_EVAL_MODE,
    "epoch_dev_mode_label": RUN_DATA_EPOCH_DEV_MODE_LABEL,
    "dev_eval_fraction": RUN_DATA_DEV_EVAL_FRACTION,
    "dev_eval_query_count": RUN_DATA_DEV_EVAL_QUERY_COUNT,
    "dev_eval_seed": RUN_DATA_DEV_EVAL_SEED,
    "train_triples_sample": TRAIN_TRIPLES_SAMPLE,
    "sampled_train_triples_path": str(sampled_train_triples_path),
    "passage_token_shards_index_path": str(passage_token_shards_index_path),
    "passage_token_store_stats_path": str(passage_token_store_stats_path),
    "counts": {
        "sampled_train_triples": sampled_train_triples_count,
        "sampled_train_queries": sampled_train_queries_count,
        "sampled_train_passages": sampled_train_passages_count,
        "epoch_dev_queries": epoch_dev_queries_count,
        "epoch_dev_candidate_rows": epoch_dev_candidate_rows_count,
    },
    "ablation_notebook": IS_ABLATION_NOTEBOOK,
    "ablation_variant": ABLATION_OUTPUT_DIRNAME,
    "freeze_token_embeddings": FREEZE_TOKEN_EMBEDDINGS,
    "freeze_segment_embeddings": FREEZE_SEGMENT_EMBEDDINGS,
    "best_dev_mrr10_checkpoint_path": str(BEST_DEV_MRR10_CHECKPOINT_PATH),
    "best_mrr_checkpoint_path": str(BEST_MRR_CHECKPOINT_PATH),
    "best_train_loss_checkpoint_path": str(BEST_TRAIN_LOSS_CHECKPOINT_PATH),
    "last_checkpoint_path": str(LAST_CHECKPOINT_PATH),
    "dev_metrics_by_step_path": str(DEV_METRICS_BY_STEP_PATH),
    "final_metrics_path": str(FINAL_METRICS_PATH),
}
print(json.dumps(prep_summary, indent=2))
prep_summary


In [ ]:
def encode_pair(query_tokens: List[int], passage_tokens: List[int], seq_len: int, max_query_len: int, max_doc_len: int) -> Dict[str, List[int]]:
    q_tokens = list(query_tokens[:max_query_len])
    d_tokens = list(passage_tokens[:max_doc_len])

    input_ids = [CLS_ID] + q_tokens + [SEP_ID] + d_tokens + [SEP_ID]
    token_type_ids = [0] + [1] * len(q_tokens) + [0] + [2] * len(d_tokens) + [0]
    attention_mask = [1] * len(input_ids)

    pad_len = seq_len - len(input_ids)
    if pad_len < 0:
        raise ValueError(f'Pair length {len(input_ids)} exceeds configured seq_len={seq_len}')

    input_ids += [PAD_ID] * pad_len
    token_type_ids += [0] * pad_len
    attention_mask += [0] * pad_len

    return {
        'input_ids': input_ids,
        'token_type_ids': token_type_ids,
        'attention_mask': attention_mask,
    }


def collate_encoded_pairs(encoded_pairs: List[Dict[str, List[int]]]) -> Dict[str, torch.Tensor]:
    batch = {
        key: torch.tensor([item[key] for item in encoded_pairs], dtype=torch.long)
        for key in ['input_ids', 'token_type_ids', 'attention_mask']
    }
    return batch


def move_batch_to_device(batch: Dict[str, torch.Tensor], device: torch.device) -> Dict[str, torch.Tensor]:
    return {
        key: value.to(device, non_blocking=True)
        for key, value in batch.items()
    }


def reciprocal_rank_at_k(ranked_pids: List[int], relevant_pids: set, k: int = 10) -> float:
    for rank, pid in enumerate(ranked_pids[:k], start=1):
        if pid in relevant_pids:
            return 1.0 / rank
    return 0.0


def iter_grouped_candidates(candidates_artifact, query_limit: int = None):
    qid_order = candidates_artifact['qid_order']
    qid_offsets = candidates_artifact['qid_offsets']
    pid_values = candidates_artifact['pid']
    bm25_ranks = candidates_artifact['bm25_rank']
    limit = len(qid_order) if query_limit is None else min(len(qid_order), query_limit)
    for idx in range(limit):
        qid = int(qid_order[idx])
        start = int(qid_offsets[idx])
        end = int(qid_offsets[idx + 1])
        yield qid, pid_values[start:end], bm25_ranks[start:end]


In [ ]:
CosSin = Tuple[torch.Tensor, torch.Tensor]


def trunc_normal_init_(tensor: torch.Tensor, std: float = 1.0, lower: float = -2.0, upper: float = 2.0):
    with torch.no_grad():
        if std == 0:
            tensor.zero_()
        else:
            sqrt2 = math.sqrt(2)
            a = math.erf(lower / sqrt2)
            b = math.erf(upper / sqrt2)
            z = (b - a) / 2

            c = (2 * math.pi) ** -0.5
            pdf_u = c * math.exp(-0.5 * lower ** 2)
            pdf_l = c * math.exp(-0.5 * upper ** 2)
            comp_std = std / math.sqrt(1 - (upper * pdf_u - lower * pdf_l) / z - ((pdf_u - pdf_l) / z) ** 2)

            tensor.uniform_(a, b)
            tensor.erfinv_()
            tensor.mul_(sqrt2 * comp_std)
            tensor.clip_(lower * comp_std, upper * comp_std)
    return tensor


def _find_multiple(a: int, b: int) -> int:
    return (-(a // -b)) * b


def rotate_half(x: torch.Tensor) -> torch.Tensor:
    x1 = x[..., : x.shape[-1] // 2]
    x2 = x[..., x.shape[-1] // 2 :]
    return torch.cat((-x2, x1), dim=-1)


def apply_rotary_pos_emb(q: torch.Tensor, k: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor):
    seq_len = q.shape[1]
    cos = cos[:seq_len]
    sin = sin[:seq_len]
    orig_dtype = q.dtype
    q = q.to(cos.dtype)
    k = k.to(cos.dtype)
    q_embed = (q * cos.unsqueeze(-2)) + (rotate_half(q) * sin.unsqueeze(-2))
    k_embed = (k * cos.unsqueeze(-2)) + (rotate_half(k) * sin.unsqueeze(-2))
    return q_embed.to(orig_dtype), k_embed.to(orig_dtype)


class CastedLinear(nn.Module):
    def __init__(self, in_features: int, out_features: int, bias: bool):
        super().__init__()
        self.weight = nn.Parameter(trunc_normal_init_(torch.empty((out_features, in_features)), std=1.0 / (in_features ** 0.5)))
        self.bias = nn.Parameter(torch.zeros((out_features,))) if bias else None

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        bias = self.bias.to(inputs.dtype) if self.bias is not None else None
        return F.linear(inputs, self.weight.to(inputs.dtype), bias=bias)


class CastedEmbedding(nn.Module):
    def __init__(self, num_embeddings: int, embedding_dim: int, init_std: float, cast_to: torch.dtype):
        super().__init__()
        self.cast_to = cast_to
        self.embedding_weight = nn.Parameter(trunc_normal_init_(torch.empty((num_embeddings, embedding_dim)), std=init_std))

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        return F.embedding(inputs, self.embedding_weight.to(self.cast_to))


class RotaryEmbedding(nn.Module):
    def __init__(self, dim: int, max_position_embeddings: int, base: float):
        super().__init__()
        inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2, dtype=torch.float32) / dim))
        t = torch.arange(max_position_embeddings, dtype=torch.float32)
        freqs = torch.outer(t, inv_freq)
        emb = torch.cat((freqs, freqs), dim=-1)
        self.register_buffer('cos_cached', emb.cos(), persistent=False)
        self.register_buffer('sin_cached', emb.sin(), persistent=False)

    def forward(self) -> CosSin:
        return self.cos_cached, self.sin_cached


class Attention(nn.Module):
    def __init__(self, hidden_size: int, head_dim: int, num_heads: int, num_key_value_heads: int, causal: bool = False):
        super().__init__()
        self.hidden_size = hidden_size
        self.head_dim = head_dim
        self.output_size = head_dim * num_heads
        self.num_heads = num_heads
        self.num_key_value_heads = num_key_value_heads
        self.causal = causal
        self.qkv_proj = CastedLinear(hidden_size, (num_heads + 2 * num_key_value_heads) * head_dim, bias=False)
        self.o_proj = CastedLinear(self.output_size, hidden_size, bias=False)

    def forward(self, cos_sin: CosSin, hidden_states: torch.Tensor, attention_mask: torch.Tensor = None) -> torch.Tensor:
        batch_size, seq_len, _ = hidden_states.shape
        qkv = self.qkv_proj(hidden_states)
        qkv = qkv.view(batch_size, seq_len, self.num_heads + 2 * self.num_key_value_heads, self.head_dim)
        query = qkv[:, :, : self.num_heads]
        key = qkv[:, :, self.num_heads : self.num_heads + self.num_key_value_heads]
        value = qkv[:, :, self.num_heads + self.num_key_value_heads :]
        if cos_sin is not None:
            cos, sin = cos_sin
            query, key = apply_rotary_pos_emb(query, key, cos, sin)
        query = query.permute(0, 2, 1, 3)
        key = key.permute(0, 2, 1, 3)
        value = value.permute(0, 2, 1, 3)
        attn_mask = attention_mask.to(query.dtype) if attention_mask is not None else None
        attn_output = scaled_dot_product_attention(query=query, key=key, value=value, attn_mask=attn_mask, is_causal=self.causal)
        attn_output = attn_output.permute(0, 2, 1, 3).reshape(batch_size, seq_len, self.output_size)
        return self.o_proj(attn_output)


class SwiGLU(nn.Module):
    def __init__(self, hidden_size: int, expansion: float):
        super().__init__()
        inter = _find_multiple(round(expansion * hidden_size * 2 / 3), 256)
        self.gate_up_proj = CastedLinear(hidden_size, inter * 2, bias=False)
        self.down_proj = CastedLinear(inter, hidden_size, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        gate, up = self.gate_up_proj(x).chunk(2, dim=-1)
        return self.down_proj(F.silu(gate) * up)


def rms_norm(hidden_states: torch.Tensor, variance_epsilon: float) -> torch.Tensor:
    input_dtype = hidden_states.dtype
    hidden_states = hidden_states.to(torch.float32)
    variance = hidden_states.square().mean(-1, keepdim=True)
    hidden_states = hidden_states * torch.rsqrt(variance + variance_epsilon)
    return hidden_states.to(input_dtype)


In [ ]:
@dataclass
class TinyRecursiveReasoningModel_ACTV1InnerCarry:
    z_H: torch.Tensor
    z_L: torch.Tensor


@dataclass
class TinyRecursiveReasoningModel_ACTV1Carry:
    inner_carry: TinyRecursiveReasoningModel_ACTV1InnerCarry
    steps: torch.Tensor
    halted: torch.Tensor
    current_data: Dict[str, torch.Tensor]


@dataclass
class TinyRecursiveReasoningModel_ACTV1Config:
    batch_size: int
    seq_len: int
    vocab_size: int
    H_cycles: int
    L_cycles: int
    H_layers: int
    L_layers: int
    hidden_size: int
    expansion: float
    num_heads: int
    pos_encodings: str
    rms_norm_eps: float = 1e-5
    rope_theta: float = 10000.0
    halt_max_steps: int = 1
    halt_exploration_prob: float = 0.0
    forward_dtype: str = 'float32'
    mlp_t: bool = False
    no_ACT_continue: bool = True
    num_segment_types: int = 3


class TinyRecursiveReasoningModel_ACTV1Block(nn.Module):
    def __init__(self, config: TinyRecursiveReasoningModel_ACTV1Config) -> None:
        super().__init__()
        self.config = config
        if self.config.mlp_t:
            self.mlp_t = SwiGLU(hidden_size=self.config.seq_len, expansion=config.expansion)
        else:
            self.self_attn = Attention(
                hidden_size=config.hidden_size,
                head_dim=config.hidden_size // config.num_heads,
                num_heads=config.num_heads,
                num_key_value_heads=config.num_heads,
                causal=False,
            )
        self.mlp = SwiGLU(hidden_size=config.hidden_size, expansion=config.expansion)
        self.norm_eps = config.rms_norm_eps

    def forward(self, cos_sin: CosSin, hidden_states: torch.Tensor, attention_mask: torch.Tensor = None) -> torch.Tensor:
        if self.config.mlp_t:
            hidden_states = hidden_states.transpose(1, 2)
            out = self.mlp_t(hidden_states)
            hidden_states = rms_norm(hidden_states + out, variance_epsilon=self.norm_eps)
            hidden_states = hidden_states.transpose(1, 2)
        else:
            hidden_states = rms_norm(
                hidden_states + self.self_attn(cos_sin=cos_sin, hidden_states=hidden_states, attention_mask=attention_mask),
                variance_epsilon=self.norm_eps,
            )
        out = self.mlp(hidden_states)
        hidden_states = rms_norm(hidden_states + out, variance_epsilon=self.norm_eps)
        return hidden_states


class TinyRecursiveReasoningModel_ACTV1ReasoningModule(nn.Module):
    def __init__(self, layers: List[TinyRecursiveReasoningModel_ACTV1Block]):
        super().__init__()
        self.layers = nn.ModuleList(layers)

    def forward(self, hidden_states: torch.Tensor, input_injection: torch.Tensor, attention_mask: torch.Tensor = None, **kwargs) -> torch.Tensor:
        hidden_states = hidden_states + input_injection
        for layer in self.layers:
            hidden_states = layer(hidden_states=hidden_states, attention_mask=attention_mask, **kwargs)
        return hidden_states


class TinyRecursiveReasoningModel_ACTV1_Inner(nn.Module):
    def __init__(self, config: TinyRecursiveReasoningModel_ACTV1Config) -> None:
        super().__init__()
        self.config = config
        self.forward_dtype = getattr(torch, self.config.forward_dtype)
        self.embed_scale = math.sqrt(self.config.hidden_size)
        embed_init_std = 1.0 / self.embed_scale

        self.embed_tokens = CastedEmbedding(self.config.vocab_size, self.config.hidden_size, init_std=embed_init_std, cast_to=self.forward_dtype)
        self.segment_emb = CastedEmbedding(self.config.num_segment_types, self.config.hidden_size, init_std=embed_init_std, cast_to=self.forward_dtype)
        self.score_head = CastedLinear(self.config.hidden_size, 1, bias=True)
        self.q_head = CastedLinear(self.config.hidden_size, 2, bias=True)

        if self.config.pos_encodings == 'rope':
            self.rotary_emb = RotaryEmbedding(
                dim=self.config.hidden_size // self.config.num_heads,
                max_position_embeddings=self.config.seq_len,
                base=self.config.rope_theta,
            )
        elif self.config.pos_encodings == 'learned':
            self.embed_pos = CastedEmbedding(self.config.seq_len, self.config.hidden_size, init_std=embed_init_std, cast_to=self.forward_dtype)

        self.L_level = TinyRecursiveReasoningModel_ACTV1ReasoningModule(
            layers=[TinyRecursiveReasoningModel_ACTV1Block(self.config) for _ in range(self.config.L_layers)]
        )

        self.register_buffer('H_init', trunc_normal_init_(torch.empty(self.config.hidden_size, dtype=self.forward_dtype), std=1), persistent=True)
        self.register_buffer('L_init', trunc_normal_init_(torch.empty(self.config.hidden_size, dtype=self.forward_dtype), std=1), persistent=True)

        with torch.no_grad():
            self.q_head.weight.zero_()
            self.q_head.bias.fill_(-5)

    def _input_embeddings(self, input_ids: torch.Tensor, token_type_ids: torch.Tensor) -> torch.Tensor:
        embedding = self.embed_tokens(input_ids.to(torch.int64))
        embedding = embedding + self.segment_emb(token_type_ids.to(torch.int64))
        if self.config.pos_encodings == 'learned':
            embedding = 0.707106781 * (embedding + self.embed_pos.embedding_weight[: input_ids.shape[1]].to(self.forward_dtype))
        return self.embed_scale * embedding

    def _attention_mask(self, attention_mask: torch.Tensor) -> torch.Tensor:
        attention_mask = attention_mask.to(torch.bool)
        additive_mask = torch.zeros(attention_mask.shape, dtype=self.forward_dtype, device=attention_mask.device)
        additive_mask = additive_mask.masked_fill(~attention_mask, torch.finfo(self.forward_dtype).min)
        return additive_mask[:, None, None, :]

    def empty_carry(self, batch_size: int) -> TinyRecursiveReasoningModel_ACTV1InnerCarry:
        device = self.H_init.device
        return TinyRecursiveReasoningModel_ACTV1InnerCarry(
            z_H=torch.empty(batch_size, self.config.seq_len, self.config.hidden_size, dtype=self.forward_dtype, device=device),
            z_L=torch.empty(batch_size, self.config.seq_len, self.config.hidden_size, dtype=self.forward_dtype, device=device),
        )

    def reset_carry(self, reset_flag: torch.Tensor, carry: TinyRecursiveReasoningModel_ACTV1InnerCarry) -> TinyRecursiveReasoningModel_ACTV1InnerCarry:
        reset_view = reset_flag.view(-1, 1, 1)
        return TinyRecursiveReasoningModel_ACTV1InnerCarry(
            z_H=torch.where(reset_view, self.H_init, carry.z_H),
            z_L=torch.where(reset_view, self.L_init, carry.z_L),
        )

    def forward(self, carry: TinyRecursiveReasoningModel_ACTV1InnerCarry, batch: Dict[str, torch.Tensor]):
        seq_info = {
            'cos_sin': self.rotary_emb() if hasattr(self, 'rotary_emb') else None,
            'attention_mask': self._attention_mask(batch['attention_mask']),
        }
        input_embeddings = self._input_embeddings(batch['input_ids'], batch['token_type_ids'])
        z_H, z_L = carry.z_H, carry.z_L
        with torch.no_grad():
            for _ in range(self.config.H_cycles - 1):
                for _ in range(self.config.L_cycles):
                    z_L = self.L_level(z_L, z_H + input_embeddings, **seq_info)
                z_H = self.L_level(z_H, z_L, **seq_info)
        for _ in range(self.config.L_cycles):
            z_L = self.L_level(z_L, z_H + input_embeddings, **seq_info)
        z_H = self.L_level(z_H, z_L, **seq_info)

        new_carry = TinyRecursiveReasoningModel_ACTV1InnerCarry(z_H=z_H.detach(), z_L=z_L.detach())
        cls_state = z_H[:, 0]
        scores = self.score_head(cls_state).squeeze(-1)
        q_logits = self.q_head(cls_state).to(torch.float32)
        return new_carry, scores, (q_logits[..., 0], q_logits[..., 1])


class TinyRecursiveReasoningModel_ACTV1(nn.Module):
    def __init__(self, config_dict: dict):
        super().__init__()
        self.config = TinyRecursiveReasoningModel_ACTV1Config(**config_dict)
        self.inner = TinyRecursiveReasoningModel_ACTV1_Inner(self.config)

    def initial_carry(self, batch: Dict[str, torch.Tensor]) -> TinyRecursiveReasoningModel_ACTV1Carry:
        batch_size = batch['input_ids'].shape[0]
        return TinyRecursiveReasoningModel_ACTV1Carry(
            inner_carry=self.inner.empty_carry(batch_size),
            steps=torch.zeros((batch_size,), dtype=torch.int32, device=batch['input_ids'].device),
            halted=torch.ones((batch_size,), dtype=torch.bool, device=batch['input_ids'].device),
            current_data={key: torch.empty_like(value) for key, value in batch.items()},
        )

    def forward(self, carry: TinyRecursiveReasoningModel_ACTV1Carry, batch: Dict[str, torch.Tensor]):
        new_inner_carry = self.inner.reset_carry(carry.halted, carry.inner_carry)
        new_steps = torch.where(carry.halted, torch.zeros_like(carry.steps), carry.steps)
        new_current_data = {
            key: torch.where(carry.halted.view((-1,) + (1,) * (value.ndim - 1)), batch[key], value)
            for key, value in carry.current_data.items()
        }
        new_inner_carry, scores, (q_halt_logits, q_continue_logits) = self.inner(new_inner_carry, new_current_data)
        outputs = {
            'scores': scores,
            'q_halt_logits': q_halt_logits,
            'q_continue_logits': q_continue_logits,
        }
        with torch.no_grad():
            new_steps = new_steps + 1
            is_last_step = new_steps >= self.config.halt_max_steps
            halted = is_last_step
            if self.training and (self.config.halt_max_steps > 1):
                if self.config.no_ACT_continue:
                    halted = halted | (q_halt_logits > 0)
                else:
                    halted = halted | (q_halt_logits > q_continue_logits)
                min_halt_steps = (torch.rand_like(q_halt_logits) < self.config.halt_exploration_prob) * torch.randint_like(
                    new_steps, low=2, high=self.config.halt_max_steps + 1
                )
                halted = halted & (new_steps >= min_halt_steps)
                if not self.config.no_ACT_continue:
                    _, _, (next_q_halt_logits, next_q_continue_logits) = self.inner(new_inner_carry, new_current_data)
                    outputs['target_q_continue'] = torch.sigmoid(
                        torch.where(is_last_step, next_q_halt_logits, torch.maximum(next_q_halt_logits, next_q_continue_logits))
                    )
        return TinyRecursiveReasoningModel_ACTV1Carry(new_inner_carry, new_steps, halted, new_current_data), outputs


In [ ]:
class PairwiseTripleDataset(Dataset):
    def __init__(self, triples_path: Path, query_token_map: Dict[int, List[int]], passage_token_map: Dict[int, List[int]]):
        self.query_token_map = query_token_map
        self.passage_token_map = passage_token_map
        self.triples: List[Tuple[int, int, int]] = []
        with triples_path.open("r", encoding="utf-8", newline="") as handle:
            reader = csv.reader(handle, delimiter="\t")
            for row in make_tqdm(reader, desc=f"Load {triples_path.name}"):
                if len(row) >= 3:
                    self.triples.append((int(row[0]), int(row[1]), int(row[2])))

    def __len__(self) -> int:
        return len(self.triples)

    def __getitem__(self, index: int):
        qid, pos_pid, neg_pid = self.triples[index]
        if qid not in self.query_token_map:
            raise KeyError(f"Missing query tokens for qid={qid}")
        if pos_pid not in self.passage_token_map or neg_pid not in self.passage_token_map:
            raise KeyError(f"Missing cached passage tokens for one of {pos_pid}, {neg_pid}")
        return {
            "qid": qid,
            "query_tokens": self.query_token_map[qid],
            "pos_tokens": self.passage_token_map[pos_pid],
            "neg_tokens": self.passage_token_map[neg_pid],
        }


def pairwise_collate(batch_items: List[Dict[str, object]]):
    pos_pairs = [encode_pair(item["query_tokens"], item["pos_tokens"], SEQ_LEN, MAX_QUERY_LEN, MAX_DOC_LEN) for item in batch_items]
    neg_pairs = [encode_pair(item["query_tokens"], item["neg_tokens"], SEQ_LEN, MAX_QUERY_LEN, MAX_DOC_LEN) for item in batch_items]
    return collate_encoded_pairs(pos_pairs), collate_encoded_pairs(neg_pairs)


def run_model_once(model, batch: Dict[str, torch.Tensor]):
    stateful_model = unwrap_model(model)
    carry = stateful_model.initial_carry(batch)

    if stateful_model.config.halt_max_steps == 1:
        carry, outputs = model(carry, batch)
        return outputs["scores"], outputs

    outputs = None
    for _ in range(stateful_model.config.halt_max_steps):
        carry, outputs = model(carry, batch)
        if bool(carry.halted.all()):
            break

    if outputs is None:
        raise RuntimeError("Model produced no outputs")

    return outputs["scores"], outputs


def build_warmup_steps(total_steps: int, warmup_ratio: float) -> int:
    if total_steps <= 0:
        return 0
    return max(0, min(total_steps - 1, int(math.ceil(total_steps * warmup_ratio))))


def resolve_step_count(explicit_steps: Optional[int], epoch_fraction: Optional[float], steps_per_epoch: int, default_steps: Optional[int] = None) -> Optional[int]:
    if explicit_steps is not None:
        explicit_steps = int(explicit_steps)
        if explicit_steps <= 0:
            raise ValueError("explicit step count must be positive")
        return explicit_steps
    if epoch_fraction is not None:
        epoch_fraction = float(epoch_fraction)
        if epoch_fraction <= 0:
            raise ValueError("epoch fraction must be positive")
        return max(1, int(math.ceil(int(steps_per_epoch) * epoch_fraction)))
    return default_steps


def build_linear_warmup_decay_lambda(warmup_steps: int, total_steps: Optional[int]):
    def lr_lambda(current_step: int) -> float:
        if total_steps is None or total_steps <= 0:
            return 1.0
        if current_step < warmup_steps:
            return float(current_step) / float(max(1, warmup_steps))
        progress = float(current_step - warmup_steps) / float(max(1, total_steps - warmup_steps))
        return max(0.0, 1.0 - progress)

    return lr_lambda


def concat_batches(pos_batch, neg_batch):
    pos_keys = set(pos_batch)
    neg_keys = set(neg_batch)
    if pos_keys != neg_keys:
        missing_in_neg = sorted(pos_keys - neg_keys)
        missing_in_pos = sorted(neg_keys - pos_keys)
        raise KeyError(
            "pos_batch and neg_batch must contain the same keys: "
            f"missing_in_neg={missing_in_neg}, missing_in_pos={missing_in_pos}"
        )

    pair_batch = {}
    for key in pos_batch:
        pos_value = pos_batch[key]
        neg_value = neg_batch[key]
        if not isinstance(pos_value, torch.Tensor) or not isinstance(neg_value, torch.Tensor):
            raise TypeError(f"Batch key {key!r} cannot be concatenated because it is not a tensor")
        if pos_value.dim() == 0 or neg_value.dim() == 0:
            raise ValueError(f"Batch key {key!r} must have a batch dimension")
        if pos_value.shape[1:] != neg_value.shape[1:]:
            raise ValueError(
                f"Batch key {key!r} has incompatible shapes for concat: "
                f"pos={tuple(pos_value.shape)}, neg={tuple(neg_value.shape)}"
            )
        pair_batch[key] = torch.cat([pos_value, neg_value], dim=0)

    return pair_batch


def compute_pairwise_batch_metrics(model, pos_batch, neg_batch):
    pair_batch = concat_batches(pos_batch, neg_batch)

    scores, _ = run_model_once(model, pair_batch)
    pos_scores, neg_scores = scores.chunk(2, dim=0)

    margin = pos_scores - neg_scores
    loss = -F.logsigmoid(margin).mean()
    pairwise_acc = (pos_scores > neg_scores).float().mean()
    return {
        "loss": loss,
        "pairwise_acc": pairwise_acc,
        "margin": margin.mean(),
        "pos_score": pos_scores.mean(),
        "neg_score": neg_scores.mean(),
    }


class ResumeDistributedSampler(DistributedSampler):
    """DistributedSampler with a per-rank start offset for efficient mid-epoch resume."""

    def __init__(self, *args, start_index: int = 0, **kwargs):
        super().__init__(*args, **kwargs)
        self.start_index = max(0, int(start_index))

    def __iter__(self):
        indices = list(super().__iter__())
        if self.start_index:
            indices = indices[self.start_index:]
        return iter(indices)

    def __len__(self) -> int:
        return max(0, super().__len__() - self.start_index)


train_dataset = PairwiseTripleDataset(sampled_train_triples_path, train_query_tokens, train_passage_tokens)
WORLD_SIZE = WORLD_SIZE_ENV if SHOULD_USE_DDP else 1


def build_train_loader_for_epoch(epoch: int, start_batch_idx: int = 0):
    start_index = max(0, int(start_batch_idx)) * int(PER_DEVICE_BATCH_SIZE)
    sampler = ResumeDistributedSampler(
        train_dataset,
        num_replicas=WORLD_SIZE,
        rank=get_rank() if SHOULD_USE_DDP else 0,
        shuffle=True,
        seed=SEED,
        drop_last=False,
        start_index=start_index,
    )
    sampler.set_epoch(int(epoch))
    loader_kwargs = dict(
        batch_size=PER_DEVICE_BATCH_SIZE,
        sampler=sampler,
        shuffle=False,
        num_workers=NUM_WORKERS,
        collate_fn=pairwise_collate,
        pin_memory=torch.cuda.is_available(),
        drop_last=True,
    )

    if NUM_WORKERS > 0:
        loader_kwargs.update(
            persistent_workers=True,
            prefetch_factor=4,
        )

    loader = DataLoader(train_dataset, **loader_kwargs)
    return loader, sampler


train_loader, train_sampler = build_train_loader_for_epoch(epoch=0, start_batch_idx=0)

GLOBAL_BATCH_SIZE = PER_DEVICE_BATCH_SIZE * WORLD_SIZE * GRAD_ACCUM_STEPS
ESTIMATED_STEPS_PER_EPOCH = max(1, math.ceil(len(train_loader) / GRAD_ACCUM_STEPS))
TARGET_TOTAL_STEPS = ESTIMATED_STEPS_PER_EPOCH * EPOCHS

RUN_STEPS_PER_SESSION = resolve_step_count(
    RUN_TRAIN_STEPS,
    RUN_EPOCH_FRACTION,
    ESTIMATED_STEPS_PER_EPOCH,
    default_steps=ESTIMATED_STEPS_PER_EPOCH,
)
RUN_STEPS_PER_SESSION = min(int(RUN_STEPS_PER_SESSION), int(TARGET_TOTAL_STEPS))

CHECKPOINT_EVERY_STEPS = resolve_step_count(
    CHECKPOINT_EVERY_N_STEPS,
    CHECKPOINT_EPOCH_FRACTION,
    ESTIMATED_STEPS_PER_EPOCH,
    default_steps=RUN_STEPS_PER_SESSION,
)
CHECKPOINT_EVERY_STEPS = int(CHECKPOINT_EVERY_STEPS) if CHECKPOINT_EVERY_STEPS is not None else None
DEV_EVAL_EVERY_STEPS = CHECKPOINT_EVERY_STEPS


ESTIMATED_TOTAL_STEPS = int(TARGET_TOTAL_STEPS)
WARMUP_STEPS = build_warmup_steps(ESTIMATED_TOTAL_STEPS, WARMUP_RATIO)


In [ ]:
def model_device(model: nn.Module) -> torch.device:
    return next(model.parameters()).device


def get_autocast_context(device: torch.device, precision: str):
    if device.type != "cuda":
        return nullcontext()
    if precision == "bf16-mixed":
        return torch.autocast(device_type="cuda", dtype=torch.bfloat16)
    if precision == "16-mixed":
        return torch.autocast(device_type="cuda", dtype=torch.float16)
    return nullcontext()


def model_autocast_context(model: nn.Module):
    return get_autocast_context(model_device(model), EFFECTIVE_PRECISION)


def score_pid_batch(model, query_tokens: List[int], pid_batch: Iterable[int], passage_token_getter, batch_size: int):
    del batch_size
    encoded_pairs = [
        encode_pair(query_tokens, list(passage_token_getter(int(pid))), SEQ_LEN, MAX_QUERY_LEN, MAX_DOC_LEN)
        for pid in pid_batch
    ]
    batch = move_batch_to_device(collate_encoded_pairs(encoded_pairs), model_device(model))
    with torch.no_grad():
        with model_autocast_context(model):
            scores, _ = run_model_once(model, batch)
    return scores.detach().float().cpu().tolist()


RANKING_METRIC_NAMES = ["mrr@10", "hit@1", "hit@3", "hit@5", "hit@10", "ndcg@10"]


def hit_at_k(ranked_pids: List[int], relevant_pids: set, k: int) -> float:
    return 1.0 if any(pid in relevant_pids for pid in ranked_pids[:k]) else 0.0


def dcg_at_k(ranked_pids: List[int], relevant_pids: set, k: int) -> float:
    dcg = 0.0
    for rank, pid in enumerate(ranked_pids[:k], start=1):
        rel = 1.0 if pid in relevant_pids else 0.0
        if rel:
            dcg += rel / math.log2(rank + 1)
    return dcg


def ideal_dcg_at_k(num_relevant: int, k: int) -> float:
    ideal_count = min(int(num_relevant), k)
    if ideal_count <= 0:
        return 0.0
    return sum(1.0 / math.log2(rank + 1) for rank in range(1, ideal_count + 1))


def ndcg_at_k(ranked_pids: List[int], relevant_pids: set, k: int) -> float:
    idcg = ideal_dcg_at_k(len(relevant_pids), k)
    if idcg <= 0.0:
        return 0.0
    return dcg_at_k(ranked_pids, relevant_pids, k) / idcg


def ranking_metrics_at_10(ranked_pids: List[int], relevant_pids: set) -> Dict[str, float]:
    return {
        "mrr@10": reciprocal_rank_at_k(ranked_pids, relevant_pids, k=10),
        "hit@1": hit_at_k(ranked_pids, relevant_pids, k=1),
        "hit@3": hit_at_k(ranked_pids, relevant_pids, k=3),
        "hit@5": hit_at_k(ranked_pids, relevant_pids, k=5),
        "hit@10": hit_at_k(ranked_pids, relevant_pids, k=10),
        "ndcg@10": ndcg_at_k(ranked_pids, relevant_pids, k=10),
    }


def evaluate_reranker(
    model,
    candidates_artifact,
    qrels: Dict[int, set],
    query_token_map: Dict[int, List[int]],
    run_path: Path,
    passage_token_getter,
    query_limit: int = None,
):
    was_training = model.training
    model.eval()
    run_path.parent.mkdir(parents=True, exist_ok=True)
    bm25_metric_values = {name: [] for name in RANKING_METRIC_NAMES}
    trm_metric_values = {name: [] for name in RANKING_METRIC_NAMES}
    evaluated_queries = 0

    total_queries = len(candidates_artifact["qid_order"])
    if query_limit is not None:
        total_queries = min(int(query_limit), total_queries)

    with run_path.open("w", encoding="utf-8") as run_handle:
        for qid, pid_values, bm25_ranks in make_tqdm(
            iter_grouped_candidates(candidates_artifact, query_limit=query_limit),
            total=total_queries,
            desc=f"Evaluate {run_path.name}",
        ):
            qid = int(qid)
            if qid not in query_token_map or qid not in qrels:
                continue
            candidate_pids = [int(pid) for pid in pid_values]
            candidate_bm25_ranks = [int(rank) for rank in bm25_ranks]
            if not candidate_pids:
                continue
            if len(candidate_pids) != len(candidate_bm25_ranks):
                raise ValueError(f"Candidate pid count and BM25 rank count differ for qid={qid}")
            scores: List[float] = []
            for start in range(0, len(candidate_pids), EVAL_BATCH_SIZE):
                pid_batch = candidate_pids[start:start + EVAL_BATCH_SIZE]
                scores.extend(
                    score_pid_batch(
                        model,
                        query_token_map[qid],
                        pid_batch,
                        passage_token_getter,
                        EVAL_BATCH_SIZE,
                    )
                )
            reranked = sorted(zip(candidate_pids, scores), key=lambda item: item[1], reverse=True)
            reranked_pids = [pid for pid, _ in reranked]
            bm25_ranked = sorted(zip(candidate_pids, candidate_bm25_ranks), key=lambda item: item[1])
            bm25_ranked_pids = [pid for pid, _ in bm25_ranked]
            relevant_pids = qrels[qid]
            bm25_metrics = ranking_metrics_at_10(bm25_ranked_pids, relevant_pids)
            trm_metrics = ranking_metrics_at_10(reranked_pids, relevant_pids)
            for metric_name, metric_value in bm25_metrics.items():
                bm25_metric_values[metric_name].append(metric_value)
            for metric_name, metric_value in trm_metrics.items():
                trm_metric_values[metric_name].append(metric_value)
            evaluated_queries += 1
            for rank, (pid, score) in enumerate(reranked, start=1):
                run_handle.write(f"{qid} Q0 {pid} {rank} {score:.6f} TRM" + chr(10))

    if was_training:
        model.train()
    metrics = {
        "queries_evaluated": evaluated_queries,
    }
    for metric_name in RANKING_METRIC_NAMES:
        values = bm25_metric_values[metric_name]
        metrics[f"bm25_{metric_name}"] = float(np.mean(values)) if values else 0.0
    for metric_name in RANKING_METRIC_NAMES:
        values = trm_metric_values[metric_name]
        metrics[f"trm_{metric_name}"] = float(np.mean(values)) if values else 0.0
    metrics["run_path"] = str(run_path)
    print(metrics)
    return metrics


In [ ]:
TRAIN_LOG_FIELDNAMES = [
    "time",
    "epoch",
    "batch_idx",
    "global_step",
    "loss",
    "loss_ema",
    "pairwise_acc",
    "margin",
    "pos_score",
    "neg_score",
    "lr",
    "grad_norm",
]
DEV_METRICS_FIELDNAMES = [
    "epoch",
    "global_step",
    "dev_eval_mode",
    "queries_evaluated",
    "bm25_mrr@10",
    "bm25_hit@1",
    "bm25_hit@3",
    "bm25_hit@5",
    "bm25_hit@10",
    "bm25_ndcg@10",
    "trm_mrr@10",
    "trm_hit@1",
    "trm_hit@3",
    "trm_hit@5",
    "trm_hit@10",
    "trm_ndcg@10",
    "run_path",
    "metrics_path",
]
DEV_METRICS_BY_STEP_FIELDNAMES = DEV_METRICS_FIELDNAMES + [
    "dev_eval_query_limit",
    "dev_eval_every_steps",
    "best_mrr",
    "best_mrr_updated",
    "best_mrr_checkpoint_path",
]


def build_dev_eval_log_payload(
    metrics,
    dev_eval_mode: str,
    epoch: Optional[int] = None,
    global_step: Optional[int] = None,
    checkpoint_path: Optional[Path] = None,
    query_limit: Optional[int] = None,
    best_mrr_value: Optional[float] = None,
    best_mrr_updated: Optional[bool] = None,
):
    payload = {}
    if epoch is not None:
        payload["epoch"] = int(epoch)
    if global_step is not None:
        payload["global_step"] = int(global_step)
    payload["dev_eval_mode"] = dev_eval_mode
    if query_limit is not None:
        payload["dev_eval/query_limit"] = int(query_limit)
    payload["queries_evaluated"] = int(metrics["queries_evaluated"])
    for prefix in ("trm", "bm25"):
        for metric_name in RANKING_METRIC_NAMES:
            compact_name = metric_name.replace("@", "")
            payload[f"dev_eval/{prefix}_{compact_name}"] = round(float(metrics[f"{prefix}_{metric_name}"]), 6)
    if checkpoint_path is not None:
        payload["checkpoint_path"] = str(checkpoint_path)
    if best_mrr_value is not None:
        payload["best_mrr"] = round(float(best_mrr_value), 6)
    if best_mrr_updated is not None:
        payload["best_mrr_updated"] = bool(best_mrr_updated)
    return payload


def save_json(path: Path, payload) -> None:
    path.write_text(json.dumps(payload, indent=2), encoding="utf-8")


def create_train_log_writer(path: Path, append: bool = False):
    path.parent.mkdir(parents=True, exist_ok=True)
    should_append = append and path.exists() and path.stat().st_size > 0
    handle = path.open("a" if should_append else "w", encoding="utf-8", newline="")
    writer = csv.DictWriter(handle, fieldnames=TRAIN_LOG_FIELDNAMES)
    if not should_append:
        writer.writeheader()
    writer._handle = handle
    return writer


def close_train_log_writer(writer) -> None:
    if writer is not None and hasattr(writer, "_handle"):
        writer._handle.close()


def save_checkpoint(
    path,
    model,
    optimizer,
    scheduler,
    scaler,
    epoch,
    global_step,
    config,
    metrics,
):
    raw_model = unwrap_model(model)
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    steps_per_epoch = int(globals().get("ESTIMATED_STEPS_PER_EPOCH", 1))
    completed_epochs = int(global_step // max(1, steps_per_epoch))
    step_in_epoch = int(global_step % max(1, steps_per_epoch))
    best_metric_value = metrics.get("best_mrr")
    if best_metric_value is None or not np.isfinite(float(best_metric_value)):
        best_metric_value = metrics.get("best_dev_mrr10")
    best_checkpoint_path = metrics.get("best_mrr_checkpoint_path") or metrics.get("best_dev_mrr10_checkpoint_path")
    torch.save(
        {
            "model_state_dict": raw_model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict() if scheduler is not None else None,
            "scaler_state_dict": scaler.state_dict() if scaler is not None else None,

            "epoch": completed_epochs,
            "epoch_idx": int(epoch),
            "completed_epochs": completed_epochs,
            "step_in_epoch": step_in_epoch,
            "global_step": int(global_step),
            "estimated_steps_per_epoch": steps_per_epoch,
            "target_total_steps": int(globals().get("TARGET_TOTAL_STEPS", 0)),
            "run_steps_per_session": int(globals().get("RUN_STEPS_PER_SESSION", 0)),
            "checkpoint_every_steps": globals().get("CHECKPOINT_EVERY_STEPS"),
            "checkpoint_time": time.time(),
            "best_metric": best_metric_value,
            "best_checkpoint_path": best_checkpoint_path,
            "config": config,
            "metrics": metrics,
        },
        path,
    )

def load_model_weights_for_eval(model, checkpoint_path, device):
    checkpoint = torch.load(checkpoint_path, map_location=device)
    unwrap_model(model).load_state_dict(checkpoint["model_state_dict"])
    return model


def restore_training_state(model, optimizer, scheduler, scaler, checkpoint_path, device):
    checkpoint = torch.load(checkpoint_path, map_location=device)
    unwrap_model(model).load_state_dict(checkpoint["model_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    if scheduler is not None and checkpoint.get("scheduler_state_dict") is not None:
        scheduler.load_state_dict(checkpoint["scheduler_state_dict"])
    if scaler is not None and checkpoint.get("scaler_state_dict") is not None:
        scaler.load_state_dict(checkpoint["scaler_state_dict"])
    return checkpoint


def save_csv_rows(path: Path, fieldnames, rows) -> None:
    with path.open("w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        writer.writeheader()
        for row in rows:
            writer.writerow(row)


def build_epoch_eval_paths(epoch: int, mode_label: str) -> Tuple[Path, Path, Path]:
    epoch_name = f"dev_epoch_{epoch + 1:03d}_{mode_label}"
    run_path = EPOCH_EVAL_DIR / f"{epoch_name}.run"
    metrics_path = EPOCH_EVAL_DIR / f"{epoch_name}_metrics.json"
    summary_path = EPOCH_EVAL_DIR / f"{epoch_name}_summary.json"
    return run_path, metrics_path, summary_path


def build_step_eval_paths(global_step: int, query_limit: int) -> Tuple[Path, Path]:
    step_name = f"dev_step_{global_step:08d}_qlimit_{int(query_limit)}"
    run_path = STEP_EVAL_DIR / f"{step_name}.run"
    metrics_path = STEP_EVAL_DIR / f"{step_name}_metrics.json"
    return run_path, metrics_path


def build_dev_metrics_row(metrics, epoch: int, global_step: int, dev_eval_mode: str):
    row = {
        "epoch": epoch,
        "global_step": global_step,
        "dev_eval_mode": dev_eval_mode,
        "queries_evaluated": int(metrics["queries_evaluated"]),
        "run_path": str(metrics["run_path"]),
        "metrics_path": str(metrics["metrics_path"]),
    }
    for prefix in ("bm25", "trm"):
        for metric_name in RANKING_METRIC_NAMES:
            row[f"{prefix}_{metric_name}"] = float(metrics[f"{prefix}_{metric_name}"])
    return row


def run_step_dev_eval(model, epoch: int, global_step: int, query_limit: int = DEV_EVAL_QUERY_LIMIT):
    metrics = None
    step_dev_mode_label = f"step_query_limit_{int(query_limit)}"
    if is_main_process():
        eval_model = unwrap_model(model)
        run_path, metrics_path = build_step_eval_paths(global_step, query_limit)
        print({"event": "step_dev_eval_start", "global_step": int(global_step), "epoch": int(epoch) + 1, "dev_eval/query_limit": int(query_limit)})
        with torch.no_grad():
            metrics = evaluate_reranker(
                eval_model,
                epoch_dev_candidates_artifact,
                epoch_dev_qrels_artifact,
                dev_query_tokens,
                run_path=run_path,
                passage_token_getter=get_cached_passage_tokens,
                query_limit=query_limit,
            )
        metrics["epoch"] = epoch + 1
        metrics["global_step"] = global_step
        metrics["dev_eval_mode"] = step_dev_mode_label
        metrics["dev_eval_query_limit"] = int(query_limit)
        metrics["dev_eval_every_steps"] = DEV_EVAL_EVERY_STEPS
        metrics["metrics_path"] = str(metrics_path)
        save_json(metrics_path, metrics)
    return metrics, step_dev_mode_label


def run_epoch_dev_eval(model, epoch: int, global_step: int):
    metrics = None
    epoch_dev_candidates = epoch_dev_candidates_artifact
    epoch_dev_qrels = epoch_dev_qrels_artifact
    epoch_dev_mode_label = RUN_DATA_EPOCH_DEV_MODE_LABEL
    if is_main_process():
        eval_model = unwrap_model(model)
        run_path, metrics_path, _ = build_epoch_eval_paths(epoch, epoch_dev_mode_label)
        with torch.no_grad():
            metrics = evaluate_reranker(
                eval_model,
                epoch_dev_candidates,
                epoch_dev_qrels,
                dev_query_tokens,
                run_path=run_path,
                passage_token_getter=get_cached_passage_tokens,
                query_limit=None,
            )
        metrics["epoch"] = epoch + 1
        metrics["global_step"] = global_step
        metrics["dev_eval_mode"] = epoch_dev_mode_label
        metrics["metrics_path"] = str(metrics_path)
        save_json(metrics_path, metrics)
        print(build_dev_eval_log_payload(metrics, epoch_dev_mode_label, epoch=epoch + 1, global_step=global_step))
    return metrics, epoch_dev_mode_label


def select_final_eval_checkpoint_path() -> Optional[Path]:
    checkpoint_candidates = [
        BEST_MRR_CHECKPOINT_PATH,
        best_mrr_checkpoint_path,
        best_dev_mrr10_checkpoint_path,
        BEST_DEV_MRR10_CHECKPOINT_PATH,
        best_train_loss_checkpoint_path,
        BEST_TRAIN_LOSS_CHECKPOINT_PATH,
        LAST_CHECKPOINT_PATH,
    ]
    for checkpoint_path in checkpoint_candidates:
        if checkpoint_path is None:
            continue
        checkpoint_path = Path(checkpoint_path)
        if checkpoint_path.exists():
            return checkpoint_path
    return None


def run_final_full_dev_eval(model, device: torch.device):
    checkpoint_path = select_final_eval_checkpoint_path()
    if checkpoint_path is None:
        raise FileNotFoundError("No checkpoint is available for final full dev evaluation.")

    metrics = None
    if is_main_process():
        eval_model = unwrap_model(model)
        load_model_weights_for_eval(eval_model, checkpoint_path, device)
        with torch.no_grad():
            metrics = evaluate_reranker(
                eval_model,
                final_dev_candidates,
                final_dev_qrels,
                dev_query_tokens,
                run_path=FINAL_RUN_PATH,
                passage_token_getter=get_cached_passage_tokens,
                query_limit=None,
            )
        metrics["dev_eval_mode"] = "final_full"
        metrics["checkpoint_path"] = str(checkpoint_path)
        metrics["metrics_path"] = str(FINAL_METRICS_PATH)
        save_json(FINAL_METRICS_PATH, metrics)
        print(build_dev_eval_log_payload(metrics, "final_full", checkpoint_path=checkpoint_path))
    ddp_barrier()
    return metrics, checkpoint_path


def train_one_epoch(
    model,
    train_loader,
    optimizer,
    scheduler,
    device,
    epoch,
    global_step,
    max_train_steps,
    grad_accum_steps,
    max_grad_norm,
    scaler=None,
    precision="32-true",
    log_writer=None,
    loss_ema=None,
    target_global_step: Optional[int] = None,
    start_batch_idx: int = 0,
    batch_idx_offset: int = 0,
    checkpoint_every_steps: Optional[int] = None,
    checkpoint_callback=None,
    dev_eval_every_steps: Optional[int] = None,
    dev_eval_callback=None,
    best_train_loss_callback=None,
):
    model.train()
    optimizer.zero_grad(set_to_none=True)
    start_batch_idx = max(0, int(start_batch_idx))
    batch_idx_offset = max(0, int(batch_idx_offset))
    target_global_step = int(target_global_step) if target_global_step is not None else None
    progress_desc = f"Epoch {epoch + 1}/{EPOCHS}"
    if batch_idx_offset > 0:
        progress_desc += f" | resume batch {batch_idx_offset}"
    progress_bar = make_tqdm(
        enumerate(train_loader),
        total=len(train_loader),
        disable=not is_main_process(),
        desc=progress_desc,
    )
    micro_accumulator = {key: [] for key in ["loss", "pairwise_acc", "margin", "pos_score", "neg_score"]}
    step_rows = []
    stop_training = False

    for batch_idx, (pos_batch, neg_batch) in progress_bar:
        if batch_idx < start_batch_idx:
            continue

        absolute_batch_idx = batch_idx_offset + batch_idx
        pos_batch = move_batch_to_device(pos_batch, device)
        neg_batch = move_batch_to_device(neg_batch, device)
        is_last_batch = batch_idx + 1 == len(train_loader)
        should_step = ((absolute_batch_idx + 1) % grad_accum_steps == 0) or is_last_batch
        sync_context = nullcontext()
        if is_dist_available_and_initialized() and hasattr(model, "no_sync") and not should_step:
            sync_context = model.no_sync()

        with sync_context:
            with get_autocast_context(device, precision):
                batch_metrics = compute_pairwise_batch_metrics(model, pos_batch, neg_batch)
                loss = batch_metrics["loss"]
                loss_for_backward = loss / grad_accum_steps
            if scaler is not None:
                scaler.scale(loss_for_backward).backward()
            else:
                loss_for_backward.backward()

        for key in micro_accumulator:
            micro_accumulator[key].append(batch_metrics[key].detach())

        if not should_step:
            continue

        if scaler is not None:
            scaler.unscale_(optimizer)
        grad_norm = torch.nn.utils.clip_grad_norm_(unwrap_model(model).parameters(), max_grad_norm)
        if scaler is not None:
            scaler.step(optimizer)
            scaler.update()
        else:
            optimizer.step()
        if scheduler is not None:
            scheduler.step()
        optimizer.zero_grad(set_to_none=True)
        global_step += 1

        reduced_metrics = {}
        for key, values in micro_accumulator.items():
            stacked = torch.stack(values)
            reduced_metrics[key] = float(reduce_mean(stacked.mean()).item())
        grad_norm_tensor = torch.as_tensor(float(grad_norm), device=device)
        grad_norm_value = float(reduce_mean(grad_norm_tensor).item())
        lr_tensor = torch.as_tensor(float(optimizer.param_groups[0]["lr"]), device=device)
        lr_value = float(reduce_mean(lr_tensor).item())
        loss_ema = reduced_metrics["loss"] if loss_ema is None else (0.98 * loss_ema + 0.02 * reduced_metrics["loss"])

        row = {
            "time": time.time(),
            "epoch": epoch + 1,
            "batch_idx": absolute_batch_idx,
            "global_step": global_step,
            "loss": reduced_metrics["loss"],
            "loss_ema": loss_ema,
            "pairwise_acc": reduced_metrics["pairwise_acc"],
            "margin": reduced_metrics["margin"],
            "pos_score": reduced_metrics["pos_score"],
            "neg_score": reduced_metrics["neg_score"],
            "lr": lr_value,
            "grad_norm": grad_norm_value,
        }
        step_rows.append(row)

        if log_writer is not None and is_main_process():
            log_writer.writerow(row)
            log_writer._handle.flush()

        if is_main_process() and global_step % TQDM_POSTFIX_EVERY_N_STEPS == 0:
            progress_bar.set_postfix(
                {
                    "global_step": global_step,
                    "loss": f"{row['loss']:.4f}",
                    "loss_ema": f"{row['loss_ema']:.4f}",
                    "acc": f"{row['pairwise_acc']:.3f}",
                    "margin": f"{row['margin']:.3f}",
                    "lr": f"{row['lr']:.2e}",
                    "grad_norm": f"{row['grad_norm']:.3f}",
                },
                refresh=False,
            )

        if best_train_loss_callback is not None:
            best_train_loss_callback(
                epoch=epoch,
                batch_idx=absolute_batch_idx,
                global_step=global_step,
                row=row,
                loss_ema=loss_ema,
            )

        should_checkpoint = False
        if checkpoint_callback is not None:
            if checkpoint_every_steps is not None and int(checkpoint_every_steps) > 0 and global_step % int(checkpoint_every_steps) == 0:
                should_checkpoint = True
            if target_global_step is not None and global_step >= target_global_step:
                should_checkpoint = True
            if should_checkpoint:
                checkpoint_callback(
                    epoch=epoch,
                    batch_idx=absolute_batch_idx,
                    global_step=global_step,
                    row=row,
                    loss_ema=loss_ema,
                )

        should_run_dev_eval = False
        if dev_eval_callback is not None:
            if dev_eval_every_steps is not None and int(dev_eval_every_steps) > 0 and global_step % int(dev_eval_every_steps) == 0:
                should_run_dev_eval = True
            if target_global_step is not None and global_step == target_global_step:
                should_run_dev_eval = True
            if should_run_dev_eval:
                dev_eval_callback(
                    epoch=epoch,
                    batch_idx=absolute_batch_idx,
                    global_step=global_step,
                    row=row,
                    loss_ema=loss_ema,
                )

        micro_accumulator = {key: [] for key in micro_accumulator}
        if target_global_step is not None and global_step >= target_global_step:
            stop_training = True
            break
        if max_train_steps is not None and global_step >= int(max_train_steps):
            stop_training = True
            break

    if is_main_process():
        progress_bar.close()

    if step_rows:
        epoch_summary = {
            "epoch": epoch + 1,
            "epoch_idx": epoch,
            "global_step": global_step,
            "start_batch_idx": batch_idx_offset + start_batch_idx,
            "end_batch_idx": int(step_rows[-1]["batch_idx"]),
            "target_global_step": target_global_step,
            "optimizer_steps": len(step_rows),
            "loss": float(np.mean([row["loss"] for row in step_rows])),
            "loss_ema": float(step_rows[-1]["loss_ema"]),
            "pairwise_acc": float(np.mean([row["pairwise_acc"] for row in step_rows])),
            "margin": float(np.mean([row["margin"] for row in step_rows])),
            "pos_score": float(np.mean([row["pos_score"] for row in step_rows])),
            "neg_score": float(np.mean([row["neg_score"] for row in step_rows])),
            "lr": float(step_rows[-1]["lr"]),
            "grad_norm": float(np.mean([row["grad_norm"] for row in step_rows])),
            "epoch_completed": bool(step_rows[-1]["batch_idx"] + 1 >= batch_idx_offset + len(train_loader)),
        }
    else:
        epoch_summary = {
            "epoch": epoch + 1,
            "epoch_idx": epoch,
            "global_step": global_step,
            "start_batch_idx": batch_idx_offset + start_batch_idx,
            "end_batch_idx": None,
            "target_global_step": target_global_step,
            "optimizer_steps": 0,
            "loss": None,
            "loss_ema": loss_ema,
            "pairwise_acc": None,
            "margin": None,
            "pos_score": None,
            "neg_score": None,
            "lr": float(optimizer.param_groups[0]["lr"]),
            "grad_norm": None,
            "epoch_completed": False,
        }
    return epoch_summary, global_step, stop_training, loss_ema

def count_model_parameters(model) -> int:
    return sum(parameter.numel() for parameter in unwrap_model(model).parameters())


def count_trainable_parameters(model) -> int:
    return sum(parameter.numel() for parameter in unwrap_model(model).parameters() if parameter.requires_grad)


def get_trainable_parameters(model):
    return [parameter for parameter in unwrap_model(model).parameters() if parameter.requires_grad]


def resolve_token_embedding_parameter(model):
    raw_model = unwrap_model(model)
    if not hasattr(raw_model, "inner") or not hasattr(raw_model.inner, "embed_tokens"):
        raise AttributeError("Could not locate token embedding layer at model.inner.embed_tokens")
    token_embedding_layer = raw_model.inner.embed_tokens
    if not hasattr(token_embedding_layer, "embedding_weight"):
        raise AttributeError("Token embedding layer does not expose embedding_weight")
    return token_embedding_layer.embedding_weight, "inner.embed_tokens.embedding_weight"


def resolve_segment_embedding_parameter(model):
    raw_model = unwrap_model(model)
    if not hasattr(raw_model, "inner") or not hasattr(raw_model.inner, "segment_emb"):
        raise AttributeError("Could not locate segment embedding layer at model.inner.segment_emb")
    segment_embedding_layer = raw_model.inner.segment_emb
    if not hasattr(segment_embedding_layer, "embedding_weight"):
        raise AttributeError("Segment embedding layer does not expose embedding_weight")
    return segment_embedding_layer.embedding_weight, "inner.segment_emb.embedding_weight"


def apply_token_embedding_freeze(model):
    token_embedding_parameter, token_embedding_path = resolve_token_embedding_parameter(model)
    if FREEZE_TOKEN_EMBEDDINGS:
        token_embedding_parameter.requires_grad_(False)

    segment_embedding_parameter, segment_embedding_path = resolve_segment_embedding_parameter(model)
    if FREEZE_SEGMENT_EMBEDDINGS:
        raise ValueError("This ablation notebook must not freeze segment embeddings.")
    if not segment_embedding_parameter.requires_grad:
        raise RuntimeError("Segment embeddings unexpectedly became frozen in the ablation notebook.")

    return {
        "token_embedding_layer_path": token_embedding_path,
        "segment_embedding_layer_path": segment_embedding_path,
        "token_embeddings_frozen": not bool(token_embedding_parameter.requires_grad),
        "segment_embeddings_frozen": not bool(segment_embedding_parameter.requires_grad),
    }


device = setup_distributed()
WORLD_SIZE = get_world_size()
model = TinyRecursiveReasoningModel_ACTV1(model_config).to(device)
total_parameters = count_model_parameters(model)
trainable_parameters_before_freeze = count_trainable_parameters(model)
freeze_diagnostics = apply_token_embedding_freeze(model)
trainable_parameters_after_freeze = count_trainable_parameters(model)
if SHOULD_USE_DDP:
    model = DDP(
        model,
        device_ids=[get_local_rank()],
        output_device=get_local_rank(),
        find_unused_parameters=False,
    )

optimizer_parameters = get_trainable_parameters(model)
optimizer_parameter_count = sum(parameter.numel() for parameter in optimizer_parameters)
if optimizer_parameter_count != trainable_parameters_after_freeze:
    raise RuntimeError("Optimizer parameter list does not match the post-freeze trainable parameter count.")

ablation_diagnostics = {
    "ablation_notebook": IS_ABLATION_NOTEBOOK,
    "ablation_variant": ABLATION_OUTPUT_DIRNAME,
    "token_embeddings_frozen": freeze_diagnostics["token_embeddings_frozen"],
    "segment_embeddings_frozen": freeze_diagnostics["segment_embeddings_frozen"],
    "trainable_parameters_before_freeze": trainable_parameters_before_freeze,
    "trainable_parameters_after_freeze": trainable_parameters_after_freeze,
    "total_parameters": total_parameters,
    "token_embedding_layer_path": freeze_diagnostics["token_embedding_layer_path"],
}

optimizer = torch.optim.AdamW(
    optimizer_parameters,
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)
scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer,
    lr_lambda=build_linear_warmup_decay_lambda(WARMUP_STEPS, ESTIMATED_TOTAL_STEPS),
)
scaler = torch.cuda.amp.GradScaler(enabled=device.type == "cuda" and EFFECTIVE_PRECISION == "16-mixed")

training_summary = {
    "profile": RUN_PROFILE,
    "num_train_examples": len(train_dataset),
    "epochs": EPOCHS,
    "max_train_steps": MAX_TRAIN_STEPS,
    "per_device_batch_size": PER_DEVICE_BATCH_SIZE,
    "devices_requested": DEVICES,
    "world_size": WORLD_SIZE,
    "grad_accum_steps": GRAD_ACCUM_STEPS,
    "global_batch_size": GLOBAL_BATCH_SIZE,
    "estimated_steps_per_epoch": ESTIMATED_STEPS_PER_EPOCH,
    "estimated_total_steps": ESTIMATED_TOTAL_STEPS,
    "target_total_steps": TARGET_TOTAL_STEPS,
    "run_steps_per_session": RUN_STEPS_PER_SESSION,
    "checkpoint_every_steps": CHECKPOINT_EVERY_STEPS,
    "dev_eval_query_limit": DEV_EVAL_QUERY_LIMIT,
    "dev_eval_every_steps": DEV_EVAL_EVERY_STEPS,
    "run_epoch_fraction": RUN_EPOCH_FRACTION,
    "checkpoint_epoch_fraction": CHECKPOINT_EPOCH_FRACTION,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "warmup_steps": WARMUP_STEPS,
    "max_grad_norm": MAX_GRAD_NORM,
    "precision": EFFECTIVE_PRECISION,
    "run_dir": str(RUN_DIR),
    "step_eval_dir": str(STEP_EVAL_DIR),
    "resume_from_checkpoint": RESUME_FROM_CHECKPOINT,
    "resume_checkpoint_path": None if is_empty_path(RESUME_CHECKPOINT_PATH) else str(RESUME_CHECKPOINT_PATH),
    "ddp_enabled": SHOULD_USE_DDP,
    "dev_eval_mode": RUN_DATA_DEV_EVAL_MODE,
    "dev_eval_fraction": RUN_DATA_DEV_EVAL_FRACTION,
    "dev_eval_query_count": RUN_DATA_DEV_EVAL_QUERY_COUNT,
    "dev_eval_seed": RUN_DATA_DEV_EVAL_SEED,
    "run_final_full_dev": RUN_FINAL_FULL_DEV,
    "train_triples_sample": TRAIN_TRIPLES_SAMPLE,
    "prep_manifest_path": str(PREP_MANIFEST_PATH),
    "run_data_manifest_path": str(RUN_DATA_MANIFEST_PATH),
    "sampled_train_triples_path": str(sampled_train_triples_path),
    "sampled_train_triples": sampled_train_triples_count,
    "sampled_train_queries": sampled_train_queries_count,
    "sampled_train_passages": sampled_train_passages_count,
    "epoch_dev_queries": epoch_dev_queries_count,
    "epoch_dev_candidate_rows": epoch_dev_candidate_rows_count,
    "ablation_notebook": IS_ABLATION_NOTEBOOK,
    "ablation_variant": ABLATION_OUTPUT_DIRNAME,
    "freeze_token_embeddings": FREEZE_TOKEN_EMBEDDINGS,
    "freeze_segment_embeddings": FREEZE_SEGMENT_EMBEDDINGS,
    "token_embedding_layer_path": freeze_diagnostics["token_embedding_layer_path"],
    "segment_embedding_layer_path": freeze_diagnostics["segment_embedding_layer_path"],
    "trainable_parameters_before_freeze": trainable_parameters_before_freeze,
    "trainable_parameters_after_freeze": trainable_parameters_after_freeze,
    "total_parameters": total_parameters,
}
if is_main_process():
    print(json.dumps(ablation_diagnostics, indent=2))
    save_json(TRAINING_CONFIG_PATH, training_summary)
    save_json(RUN_ARTIFACTS_PATH, run_artifacts)
    print(f"Checkpoint directory: {CHECKPOINT_DIR}")
    print(json.dumps(training_summary, indent=2))

start_epoch = 0
global_step = 0
best_train_loss = float("inf")
best_mrr = float("-inf")
best_dev_mrr10 = float("-inf")
best_train_loss_step = None
best_mrr_checkpoint_path = None
best_train_loss_checkpoint_path = None
best_dev_mrr10_checkpoint_path = None
epoch_summaries = []
dev_metrics_history = []
step_dev_metrics_history = []
loss_ema = None
final_dev_metrics = None
final_eval_checkpoint_path = None

if RESUME_FROM_CHECKPOINT:
    resume_path = RESUME_CHECKPOINT_PATH if not is_empty_path(RESUME_CHECKPOINT_PATH) else LAST_CHECKPOINT_PATH
    if not resume_path.is_file():
        raise FileNotFoundError(f"Resume checkpoint not found: {resume_path}")
    if is_main_process():
        print(f"Loading resume checkpoint: {resume_path}")
    resume_checkpoint = restore_training_state(model, optimizer, scheduler, scaler if scaler.is_enabled() else None, resume_path, device)
    global_step = int(resume_checkpoint["global_step"])
    checkpoint_steps_per_epoch = resume_checkpoint.get("estimated_steps_per_epoch")
    if checkpoint_steps_per_epoch is not None and int(checkpoint_steps_per_epoch) != int(ESTIMATED_STEPS_PER_EPOCH):
        raise ValueError(
            "The checkpoint was created with estimated_steps_per_epoch="
            f"{checkpoint_steps_per_epoch}, but the current run has ESTIMATED_STEPS_PER_EPOCH={ESTIMATED_STEPS_PER_EPOCH}. "
            "Do not change train sample count, batch size, world size, or grad_accum_steps when resuming mid-epoch."
        )
    start_epoch = int(global_step // ESTIMATED_STEPS_PER_EPOCH)
    resume_metrics = resume_checkpoint.get("metrics") or {}
    best_train_loss = float(resume_metrics.get("best_train_loss", best_train_loss))
    best_train_loss_step = resume_metrics.get("best_train_loss_step")
    if best_train_loss_step is not None:
        best_train_loss_step = int(best_train_loss_step)
    resume_best_mrr = resume_metrics.get("best_mrr")
    if resume_best_mrr is None:
        resume_best_mrr = resume_metrics.get("best_dev_mrr10")
    if resume_best_mrr is None:
        resume_best_mrr = resume_checkpoint.get("best_metric")
    if resume_best_mrr is None:
        resume_best_mrr = resume_metrics.get("best_quick_mrr10", best_mrr)
    best_mrr = float(resume_best_mrr)
    resume_best_dev_mrr10 = resume_metrics.get("best_dev_mrr10")
    if resume_best_dev_mrr10 is None:
        resume_best_dev_mrr10 = resume_metrics.get("best_full_dev_mrr10")
    if resume_best_dev_mrr10 is None:
        resume_best_dev_mrr10 = resume_checkpoint.get("best_metric")
    if resume_best_dev_mrr10 is None:
        resume_best_dev_mrr10 = resume_metrics.get("best_quick_mrr10", best_dev_mrr10)
    best_dev_mrr10 = float(resume_best_dev_mrr10)
    best_train_loss_checkpoint_path = resume_metrics.get("best_train_loss_checkpoint_path")
    best_mrr_checkpoint_path = (
        resume_metrics.get("best_mrr_checkpoint_path")
        or resume_metrics.get("best_dev_mrr10_checkpoint_path")
        or resume_checkpoint.get("best_checkpoint_path")
    )
    best_dev_mrr10_checkpoint_path = resume_metrics.get("best_dev_mrr10_checkpoint_path") or best_mrr_checkpoint_path
    loss_ema = resume_metrics.get("loss_ema")
    if ALLOW_RESUME_LR_OVERRIDE:
        for param_group in optimizer.param_groups:
            param_group["lr"] = LEARNING_RATE
            param_group["initial_lr"] = LEARNING_RATE
        if scheduler is not None:
            scheduler.base_lrs = [LEARNING_RATE for _ in optimizer.param_groups]
    if is_main_process():
        print(f"Resuming from global_step={global_step}")
        print(f"Completed epochs: {global_step / ESTIMATED_STEPS_PER_EPOCH:.3f} / {EPOCHS}")
else:
    start_epoch = 0

ddp_barrier()

train_log_writer = create_train_log_writer(TRAIN_LOG_PATH, append=RESUME_FROM_CHECKPOINT) if is_main_process() else None
stop_training = False
run_start_global_step = int(global_step)
run_target_global_step = min(int(TARGET_TOTAL_STEPS), run_start_global_step + int(RUN_STEPS_PER_SESSION))

if is_main_process():
    print(
        "Step-based training window: "
        f"global_step {run_start_global_step} -> {run_target_global_step} / {TARGET_TOTAL_STEPS} "
        f"({run_target_global_step / ESTIMATED_STEPS_PER_EPOCH:.3f}/{EPOCHS} epochs)"
    )
    print(f"Periodic checkpoint every {CHECKPOINT_EVERY_STEPS} optimizer steps; last checkpoint: {LAST_CHECKPOINT_PATH}")
    print(f"Step dev eval every {DEV_EVAL_EVERY_STEPS} optimizer steps; query_limit={DEV_EVAL_QUERY_LIMIT}")


def make_checkpoint_metrics(epoch_summary=None, step_row=None, loss_ema_value=None, dev_metrics=None, dev_mode_label=None):
    return {
        "epoch_summary": epoch_summary,
        "step_row": step_row,
        "best_train_loss": best_train_loss,
        "best_train_loss_step": best_train_loss_step,
        "best_mrr": best_mrr,
        "best_mrr_metric_name": "trm_mrr@10",
        "best_mrr_checkpoint_path": best_mrr_checkpoint_path,
        "best_dev_mrr10": best_dev_mrr10,
        "best_train_loss_checkpoint_path": best_train_loss_checkpoint_path,
        "best_dev_mrr10_checkpoint_path": best_dev_mrr10_checkpoint_path,
        "loss_ema": loss_ema if loss_ema_value is None else loss_ema_value,
        "dev_eval_metrics": dev_metrics,
        "dev_eval_mode": dev_mode_label,
        "run_start_global_step": run_start_global_step,
        "run_target_global_step": run_target_global_step,
        "target_total_steps": TARGET_TOTAL_STEPS,
        "estimated_steps_per_epoch": ESTIMATED_STEPS_PER_EPOCH,
        "dev_eval_query_limit": DEV_EVAL_QUERY_LIMIT,
        "dev_eval_every_steps": DEV_EVAL_EVERY_STEPS,
    }


def best_train_loss_checkpoint_callback(epoch, batch_idx, global_step, row, loss_ema):
    global best_train_loss, best_train_loss_step, best_train_loss_checkpoint_path

    step_loss = row.get("loss")
    if step_loss is None:
        return
    step_loss = float(step_loss)
    if not np.isfinite(step_loss) or step_loss >= best_train_loss:
        return

    best_train_loss = step_loss
    best_train_loss_step = int(global_step)
    best_train_loss_checkpoint_path = str(BEST_TRAIN_LOSS_CHECKPOINT_PATH)

    if not is_main_process():
        return

    checkpoint_metrics = make_checkpoint_metrics(
        step_row=row,
        loss_ema_value=loss_ema,
        dev_mode_label="not_run_step_best_train_loss",
    )
    save_checkpoint(
        BEST_TRAIN_LOSS_CHECKPOINT_PATH,
        model,
        optimizer,
        scheduler,
        scaler if scaler.is_enabled() else None,
        epoch,
        global_step,
        training_summary,
        checkpoint_metrics,
    )
    print(
        f"Updated best train loss checkpoint: {BEST_TRAIN_LOSS_CHECKPOINT_PATH} "
        f"(loss={best_train_loss:.6f}, global_step={global_step})"
    )


def maybe_update_best_mrr_checkpoint(model, optimizer, scheduler, scaler, epoch, global_step, step_row, loss_ema, dev_metrics):
    global best_mrr, best_mrr_checkpoint_path

    if dev_metrics is None or "trm_mrr@10" not in dev_metrics:
        return False
    step_mrr = float(dev_metrics["trm_mrr@10"])
    if not np.isfinite(step_mrr) or step_mrr <= best_mrr:
        return False

    best_mrr = step_mrr
    best_mrr_checkpoint_path = str(BEST_MRR_CHECKPOINT_PATH)
    checkpoint_metrics = make_checkpoint_metrics(
        step_row=step_row,
        loss_ema_value=loss_ema,
        dev_metrics=dev_metrics,
        dev_mode_label=dev_metrics.get("dev_eval_mode"),
    )
    checkpoint_metrics.update(
        {
            "best_mrr": best_mrr,
            "best_mrr_metric_name": "trm_mrr@10",
            "best_mrr_checkpoint_path": best_mrr_checkpoint_path,
            "dev_eval_query_limit": DEV_EVAL_QUERY_LIMIT,
            "dev_eval_every_steps": DEV_EVAL_EVERY_STEPS,
            "dev_eval_metrics": dev_metrics,
            "global_step": int(global_step),
            "epoch": int(epoch) + 1,
            "best_mrr_updated": True,
        }
    )
    save_checkpoint(
        BEST_MRR_CHECKPOINT_PATH,
        model,
        optimizer,
        scheduler,
        scaler if scaler.is_enabled() else None,
        epoch,
        global_step,
        training_summary,
        checkpoint_metrics,
    )
    print(
        f"Updated best MRR checkpoint: {BEST_MRR_CHECKPOINT_PATH} "
        f"(trm_mrr@10={best_mrr:.6f}, global_step={global_step})"
    )
    return True


def step_dev_eval_callback(epoch, batch_idx, global_step, row, loss_ema):
    if is_main_process():
        dev_metrics, dev_mode_label = run_step_dev_eval(
            model,
            epoch=epoch,
            global_step=global_step,
            query_limit=DEV_EVAL_QUERY_LIMIT,
        )
        best_mrr_updated = maybe_update_best_mrr_checkpoint(
            model,
            optimizer,
            scheduler,
            scaler,
            epoch,
            global_step,
            row,
            loss_ema,
            dev_metrics,
        )
        if dev_metrics is not None:
            dev_metrics_row = build_dev_metrics_row(dev_metrics, epoch + 1, global_step, dev_mode_label)
            dev_metrics_row.update(
                {
                    "dev_eval_query_limit": DEV_EVAL_QUERY_LIMIT,
                    "dev_eval_every_steps": DEV_EVAL_EVERY_STEPS,
                    "best_mrr": best_mrr,
                    "best_mrr_updated": bool(best_mrr_updated),
                    "best_mrr_checkpoint_path": best_mrr_checkpoint_path,
                }
            )
            step_dev_metrics_history.append(dev_metrics_row)
            save_csv_rows(DEV_METRICS_BY_STEP_PATH, DEV_METRICS_BY_STEP_FIELDNAMES, step_dev_metrics_history)
            print(
                build_dev_eval_log_payload(
                    dev_metrics,
                    dev_mode_label,
                    epoch=epoch + 1,
                    global_step=global_step,
                    checkpoint_path=BEST_MRR_CHECKPOINT_PATH if best_mrr_updated else None,
                    query_limit=DEV_EVAL_QUERY_LIMIT,
                    best_mrr_value=best_mrr,
                    best_mrr_updated=best_mrr_updated,
                )
            )
    ddp_barrier()


def periodic_checkpoint_callback(epoch, batch_idx, global_step, row, loss_ema):
    if not is_main_process():
        return
    checkpoint_metrics = make_checkpoint_metrics(step_row=row, loss_ema_value=loss_ema, dev_mode_label="not_run_step_checkpoint")
    step_checkpoint_path = CHECKPOINT_DIR / f"step_{global_step:08d}.pt"
    save_checkpoint(
        step_checkpoint_path,
        model,
        optimizer,
        scheduler,
        scaler if scaler.is_enabled() else None,
        epoch,
        global_step,
        training_summary,
        checkpoint_metrics,
    )
    save_checkpoint(
        LAST_CHECKPOINT_PATH,
        model,
        optimizer,
        scheduler,
        scaler if scaler.is_enabled() else None,
        epoch,
        global_step,
        training_summary,
        checkpoint_metrics,
    )
    print(f"Saved step checkpoint: {step_checkpoint_path}")
    print(f"Updated last checkpoint: {LAST_CHECKPOINT_PATH}")


try:
    while global_step < int(TARGET_TOTAL_STEPS) and global_step < int(run_target_global_step):
        epoch = int(global_step // ESTIMATED_STEPS_PER_EPOCH)
        step_in_epoch = int(global_step % ESTIMATED_STEPS_PER_EPOCH)
        start_batch_idx = min(len(train_loader), step_in_epoch * GRAD_ACCUM_STEPS)
        segment_train_loader, segment_train_sampler = build_train_loader_for_epoch(epoch=epoch, start_batch_idx=start_batch_idx)

        epoch_summary, global_step, stop_training, loss_ema = train_one_epoch(
            model=model,
            train_loader=segment_train_loader,
            optimizer=optimizer,
            scheduler=scheduler,
            device=device,
            epoch=epoch,
            global_step=global_step,
            max_train_steps=MAX_TRAIN_STEPS,
            grad_accum_steps=GRAD_ACCUM_STEPS,
            max_grad_norm=MAX_GRAD_NORM,
            scaler=scaler if scaler.is_enabled() else None,
            precision=EFFECTIVE_PRECISION,
            log_writer=train_log_writer,
            loss_ema=loss_ema,
            target_global_step=run_target_global_step,
            start_batch_idx=0,
            batch_idx_offset=start_batch_idx,
            checkpoint_every_steps=CHECKPOINT_EVERY_STEPS,
            checkpoint_callback=periodic_checkpoint_callback,
            dev_eval_every_steps=DEV_EVAL_EVERY_STEPS,
            dev_eval_callback=step_dev_eval_callback,
            best_train_loss_callback=best_train_loss_checkpoint_callback,
        )
        epoch_summary["loss_ema"] = loss_ema
        epoch_summary["run_start_global_step"] = run_start_global_step
        epoch_summary["run_target_global_step"] = run_target_global_step
        epoch_summary["target_total_steps"] = TARGET_TOTAL_STEPS
        epoch_summary["completed_epochs_float"] = float(global_step / ESTIMATED_STEPS_PER_EPOCH)

        completed_epoch_boundary = global_step > 0 and global_step % ESTIMATED_STEPS_PER_EPOCH == 0
        reached_total_target = global_step >= int(TARGET_TOTAL_STEPS)
        should_run_epoch_eval = completed_epoch_boundary or reached_total_target or RUN_EPOCH_DEV_EVAL_ON_PARTIAL

        epoch_dev_metrics = None
        epoch_dev_mode_label = "skipped_partial_step_chunk"
        if should_run_epoch_eval:
            epoch_dev_metrics, epoch_dev_mode_label = run_epoch_dev_eval(model, epoch, global_step)
        epoch_summary["dev_eval_mode"] = epoch_dev_mode_label
        if epoch_dev_metrics is not None:
            epoch_summary["dev_mrr10"] = float(epoch_dev_metrics["trm_mrr@10"])
            epoch_summary["dev_bm25_mrr10"] = float(epoch_dev_metrics["bm25_mrr@10"])
            epoch_summary["dev_queries_evaluated"] = int(epoch_dev_metrics["queries_evaluated"])
            epoch_summary["dev_run_path"] = str(epoch_dev_metrics["run_path"])
            epoch_summary["dev_metrics_path"] = str(epoch_dev_metrics["metrics_path"])
            dev_metrics_row = {
                "epoch": epoch + 1,
                "global_step": global_step,
                "dev_eval_mode": epoch_dev_mode_label,
                "queries_evaluated": int(epoch_dev_metrics["queries_evaluated"]),
                "run_path": str(epoch_dev_metrics["run_path"]),
                "metrics_path": str(epoch_dev_metrics["metrics_path"]),
            }
            for prefix in ("bm25", "trm"):
                for metric_name in RANKING_METRIC_NAMES:
                    dev_metrics_row[f"{prefix}_{metric_name}"] = float(epoch_dev_metrics[f"{prefix}_{metric_name}"])
            dev_metrics_history.append(dev_metrics_row)
        epoch_summaries.append(epoch_summary)

        improved_dev_mrr10 = epoch_dev_metrics is not None and float(epoch_dev_metrics["trm_mrr@10"]) > best_dev_mrr10
        if improved_dev_mrr10:
            best_dev_mrr10 = float(epoch_dev_metrics["trm_mrr@10"])
        if improved_dev_mrr10:
            best_dev_mrr10_checkpoint_path = str(BEST_DEV_MRR10_CHECKPOINT_PATH)

        if is_main_process():
            if epoch_dev_metrics is not None:
                _, _, epoch_summary_path = build_epoch_eval_paths(epoch, epoch_dev_mode_label)
            else:
                epoch_summary_path = LOG_DIR / f"step_{global_step:08d}_summary.json"
            save_json(
                epoch_summary_path,
                {
                    "epoch": epoch + 1,
                    "global_step": global_step,
                    "train": epoch_summary,
                    "dev_eval": epoch_dev_metrics,
                },
            )
            save_json(EPOCH_SUMMARIES_PATH, epoch_summaries)
            save_csv_rows(DEV_METRICS_BY_EPOCH_PATH, DEV_METRICS_FIELDNAMES, dev_metrics_history)
            checkpoint_metrics = make_checkpoint_metrics(
                epoch_summary=epoch_summary,
                loss_ema_value=loss_ema,
                dev_metrics=epoch_dev_metrics,
                dev_mode_label=epoch_dev_mode_label,
            )
            if completed_epoch_boundary:
                segment_checkpoint_path = CHECKPOINT_DIR / f"epoch_{global_step // ESTIMATED_STEPS_PER_EPOCH:03d}.pt"
            else:
                segment_checkpoint_path = CHECKPOINT_DIR / f"step_{global_step:08d}.pt"
            save_checkpoint(
                segment_checkpoint_path,
                model,
                optimizer,
                scheduler,
                scaler if scaler.is_enabled() else None,
                epoch,
                global_step,
                training_summary,
                checkpoint_metrics,
            )
            save_checkpoint(
                LAST_CHECKPOINT_PATH,
                model,
                optimizer,
                scheduler,
                scaler if scaler.is_enabled() else None,
                epoch,
                global_step,
                training_summary,
                checkpoint_metrics,
            )
            if improved_dev_mrr10:
                save_checkpoint(
                    BEST_DEV_MRR10_CHECKPOINT_PATH,
                    model,
                    optimizer,
                    scheduler,
                    scaler if scaler.is_enabled() else None,
                    epoch,
                    global_step,
                    training_summary,
                    checkpoint_metrics,
                )
            print(f"Saved segment checkpoint: {segment_checkpoint_path}")
            print(f"Updated last checkpoint: {LAST_CHECKPOINT_PATH}")
        ddp_barrier()
        if epoch_summary.get("optimizer_steps", 0) == 0:
            if is_main_process():
                print("No optimizer steps were completed in this segment; stopping to avoid an infinite loop.")
            break
        if stop_training:
            break
finally:
    close_train_log_writer(train_log_writer)

if RUN_FINAL_FULL_DEV and global_step >= int(TARGET_TOTAL_STEPS):
    final_dev_metrics, final_eval_checkpoint_path = run_final_full_dev_eval(model, device)
elif RUN_FINAL_FULL_DEV and is_main_process():
    print(
        "Skipping final full dev eval until training reaches target_total_steps: "
        f"global_step={global_step}, target_total_steps={TARGET_TOTAL_STEPS}"
    )

fit_summary = {
    "global_step": int(global_step),
    "training_complete": bool(global_step >= int(TARGET_TOTAL_STEPS)),
    "epochs_completed": int(global_step // ESTIMATED_STEPS_PER_EPOCH),
    "epochs_completed_float": float(global_step / ESTIMATED_STEPS_PER_EPOCH),
    "segments_completed_this_run": len(epoch_summaries),
    "target_total_steps": int(TARGET_TOTAL_STEPS),
    "run_start_global_step": int(run_start_global_step),
    "run_target_global_step": int(run_target_global_step),
    "run_steps_per_session": int(RUN_STEPS_PER_SESSION),
    "checkpoint_every_steps": int(CHECKPOINT_EVERY_STEPS) if CHECKPOINT_EVERY_STEPS is not None else None,
    "dev_eval_query_limit": int(DEV_EVAL_QUERY_LIMIT) if DEV_EVAL_QUERY_LIMIT is not None else None,
    "dev_eval_every_steps": int(DEV_EVAL_EVERY_STEPS) if DEV_EVAL_EVERY_STEPS is not None else None,
    "checkpoint_dir": str(CHECKPOINT_DIR),
    "log_dir": str(LOG_DIR),
    "eval_dir": str(EVAL_DIR),
    "epoch_eval_dir": str(EPOCH_EVAL_DIR),
    "step_eval_dir": str(STEP_EVAL_DIR),
    "final_eval_dir": str(FINAL_EVAL_DIR),
    "run_artifacts_path": str(RUN_ARTIFACTS_PATH),
    "train_log_path": str(TRAIN_LOG_PATH),
    "dev_metrics_by_epoch_path": str(DEV_METRICS_BY_EPOCH_PATH),
    "dev_metrics_by_step_path": str(DEV_METRICS_BY_STEP_PATH),
    "last_checkpoint_path": str(LAST_CHECKPOINT_PATH) if LAST_CHECKPOINT_PATH.exists() else None,
    "best_mrr": None if not np.isfinite(best_mrr) else float(best_mrr),
    "best_mrr_checkpoint_path": str(BEST_MRR_CHECKPOINT_PATH) if BEST_MRR_CHECKPOINT_PATH.exists() else None,
    "best_train_loss": None if not np.isfinite(best_train_loss) else float(best_train_loss),
    "best_train_loss_step": best_train_loss_step,
    "best_train_loss_checkpoint_path": str(BEST_TRAIN_LOSS_CHECKPOINT_PATH) if BEST_TRAIN_LOSS_CHECKPOINT_PATH.exists() else None,
    "best_dev_mrr10_checkpoint_path": str(BEST_DEV_MRR10_CHECKPOINT_PATH) if BEST_DEV_MRR10_CHECKPOINT_PATH.exists() else None,
    "final_eval_checkpoint_path": str(final_eval_checkpoint_path) if final_eval_checkpoint_path is not None else None,
    "final_metrics_path": str(FINAL_METRICS_PATH) if FINAL_METRICS_PATH.exists() else None,
    "dev_eval_mode": RUN_DATA_DEV_EVAL_MODE,
    "dev_eval_fraction": RUN_DATA_DEV_EVAL_FRACTION,
    "dev_eval_query_count": RUN_DATA_DEV_EVAL_QUERY_COUNT,
    "dev_eval_seed": RUN_DATA_DEV_EVAL_SEED,
    "epoch_dev_mode_label": RUN_DATA_EPOCH_DEV_MODE_LABEL,
}
if final_dev_metrics is not None:
    fit_summary["final_full_dev_mrr10"] = float(final_dev_metrics["trm_mrr@10"])
    fit_summary["final_full_dev_bm25_mrr10"] = float(final_dev_metrics["bm25_mrr@10"])
    fit_summary["final_full_dev_queries_evaluated"] = int(final_dev_metrics["queries_evaluated"])
if is_main_process():
    save_json(FIT_SUMMARY_PATH, fit_summary)
    print(json.dumps(fit_summary, indent=2))
cleanup_distributed()
fit_summary
